# Preparação dos dados selecionados — cheias do Guaíba de 2024

Este notebook documenta a preparação das bases selecionadas a partir da [exploração inicial](processamento_cheias_guaiba_2024.ipynb): bairros, escolas, cadastro SUS, atenção primária e quatro manchas históricas estimadas de inundação em Porto Alegre.

O conteúdo reúne verificações, seleção de atributos, correspondência de nomes, tratamentos documentados, cálculo de áreas e exportação de nove arquivos GeoParquet em `dados/tratados/`. As fontes de `dados/brutos/` são preservadas. Possibilidades de análise futura são apresentadas separadamente dos procedimentos executados.


## 1. Bibliotecas e localização dos arquivos

Pandas e GeoPandas apoiam a preparação tabular e espacial; Pyogrio, a leitura do GeoPackage; Shapely, as operações geométricas; e PyArrow, a exportação em GeoParquet. As bibliotecas padrão `pathlib`, `json` e `sqlite3` permitem trabalhar com caminhos, cenários e consultas ao banco.

Os caminhos pressupõem execução a partir de `notebooks/cheias_guaiba/`. `Path.cwd().resolve().parents[1]` sobe dois níveis e define a raiz do projeto. O JSON do INCT está em `dados/data.json`. Nenhuma base é carregada ou alterada nesta etapa.


In [1]:
from pathlib import Path
import json
import sqlite3

import pandas as pd
import geopandas as gpd
import pyogrio
import shapely
import pyarrow

In [2]:
RAIZ = Path.cwd().resolve().parents[1]

DADOS = RAIZ / "dados"
DADOS_BRUTOS = DADOS / "brutos"
DADOS_TRATADOS = DADOS / "tratados"

ARQUIVO_BANCO = DADOS_BRUTOS / "cheias_rhguaiba_2024_db_v5.gpkg"
ARQUIVO_GLOSSARIO = DADOS / "cheias_rhguaiba_2024_db_v5_glossario.csv"
ARQUIVO_JSON_INCT = DADOS / "data.json"

arquivos_entrada = {
    "GeoPackage": ARQUIVO_BANCO,
    "Glossário": ARQUIVO_GLOSSARIO,
    "JSON do INCT": ARQUIVO_JSON_INCT,
}

for nome, caminho in arquivos_entrada.items():
    if not caminho.is_file():
        raise FileNotFoundError(f"{nome} não encontrado: {caminho}")
    print(f"{nome}: {caminho}")



GeoPackage: /home/lucas/CDIA/CPA/ProjetoExtensionista-CPA/dados/brutos/cheias_rhguaiba_2024_db_v5.gpkg
Glossário: /home/lucas/CDIA/CPA/ProjetoExtensionista-CPA/dados/cheias_rhguaiba_2024_db_v5_glossario.csv
JSON do INCT: /home/lucas/CDIA/CPA/ProjetoExtensionista-CPA/dados/data.json


## 2. Referência territorial — bairros

**Camada:** `poa_admin_bairros`.

A preparação inclui verificações geométricas e de identificadores, correspondência dos 83 registros territoriais com os 94 bairros do JSON, seleção de atributos e exportação.


### Leitura da camada

A leitura utiliza a camada `poa_admin_bairros`. As verificações seguintes não corrigem ou excluem registros.


In [3]:
bairros = gpd.read_file(
    ARQUIVO_BANCO,
    layer="poa_admin_bairros"
)

display(bairros)

,nmbop,cdbop,nmrop,cdrop,nmrp,cdrp,cota_mean,geometry
0,AGRONOMIA,BOP71,LOMBA DO PINHEIRO,OP04,LOMBA DO PINHEIRO - PARTENON,RP07,125.157306,"MULTIPOLYGON Z (((489305.255 6676873.088 0, 48..."
1,ANCHIETA,BOP19,HUMAITA-NAVEGANTES,OP01,HUMAITA - NAVEGANTES - ILHAS E NOROESTE,RP02,1.954820,"MULTIPOLYGON Z (((484961.018 6685092.233 0, 48..."
2,ARQUIPELAGO,BOP36,ILHAS,OP17,HUMAITA - NAVEGANTES - ILHAS E NOROESTE,RP02,1.162602,"MULTIPOLYGON Z (((471983.597 6674237.441 0, 47..."
3,AUXILIADORA,BOP01,CENTRO,OP16,CENTRO,RP01,28.703022,"MULTIPOLYGON Z (((482207 6678761.422 0, 482243..."
4,AZENHA,BOP02,CENTRO,OP16,CENTRO,RP01,12.897710,"MULTIPOLYGON Z (((479842.718 6675588.541 0, 47..."
...,...,...,...,...,...,...,...,...
78,SANTA MARIA GORETTI,BOP32,NOROESTE,OP02,HUMAITA - NAVEGANTES - ILHAS E NOROESTE,RP02,4.145914,"MULTIPOLYGON Z (((483675.3 6680321.662 0, 4836..."
79,CAVALHADA,BOP57,CENTRO SUL,OP12,CENTRO-SUL E SUL,RP06,23.222729,"MULTIPOLYGON Z (((478348.363 6670374.305 0, 47..."
80,PRAIA DE BELAS,BOP15,CENTRO,OP16,CENTRO,RP01,2.401748,"MULTIPOLYGON Z (((477146.457 6677062.151 0, 47..."
81,PONTA GROSSA,BOP83,EXTREMO SUL,OP13,RESTINGA - EXTREMO SUL,RP08,14.849819,"MULTIPOLYGON Z (((482726.295 6662157.991 0, 48..."


### Identificadores e geometrias

São verificadas ausências e repetições nos nomes e códigos, além de geometrias nulas, vazias e inválidas. A validade geométrica não comprova precisão ou cobertura territorial.


In [4]:
print(f"Quantidade de registros: {len(bairros)}")
print(f"CRS: {bairros.crs}")

print("\nAusências nos identificadores:")
display(bairros[["nmbop", "cdbop"]].isna().sum())

print("\nNomes repetidos:")
display(bairros.loc[bairros["nmbop"].duplicated(keep=False)])

print("\nCódigos repetidos:")
display(bairros.loc[bairros["cdbop"].duplicated(keep=False)])

geometria_nula = bairros.geometry.isna()
geometria_vazia = bairros.geometry.is_empty
geometria_invalida = (
    ~geometria_nula
    & ~geometria_vazia
    & ~bairros.geometry.is_valid
)

print(f"\nGeometrias nulas: {geometria_nula.sum()}")
print(f"Geometrias vazias: {geometria_vazia.sum()}")
print(f"Geometrias inválidas não vazias: {geometria_invalida.sum()}")

display(bairros.loc[
    geometria_nula | geometria_vazia | geometria_invalida
])

Quantidade de registros: 83
CRS: EPSG:31982

Ausências nos identificadores:


nmbop    0
cdbop    0
dtype: int64


Nomes repetidos:


,nmbop,cdbop,nmrop,cdrop,nmrp,cdrp,cota_mean,geometry



Códigos repetidos:


,nmbop,cdbop,nmrop,cdrop,nmrp,cdrp,cota_mean,geometry



Geometrias nulas: 0
Geometrias vazias: 0
Geometrias inválidas não vazias: 0


,nmbop,cdbop,nmrop,cdrop,nmrp,cdrp,cota_mean,geometry


### Nomes dos bairros nos cenários do INCT

O primeiro cenário é a referência para a comparação dos nomes presentes nos demais. Diferenças são apresentadas para investigação, sem alterar o JSON.


In [5]:
with ARQUIVO_JSON_INCT.open(encoding="utf-8") as arquivo:
    cenarios = json.load(arquivo)

primeiro_cenario = next(iter(cenarios))
nomes_json = set(cenarios[primeiro_cenario])

diferencas_entre_cenarios = {}

for nome_cenario, registros in cenarios.items():
    nomes_cenario = set(registros)

    if nomes_cenario != nomes_json:
        diferencas_entre_cenarios[nome_cenario] = {
            "somente_neste_cenario": sorted(nomes_cenario - nomes_json),
            "ausentes_neste_cenario": sorted(nomes_json - nomes_cenario),
        }

print(f"Cenário de referência: {primeiro_cenario}")
print(f"Bairros no cenário: {len(nomes_json)}")
print(f"Cenários com nomes diferentes: {len(diferencas_entre_cenarios)}")

display(diferencas_entre_cenarios)

Cenário de referência: 3_0m
Bairros no cenário: 94
Cenários com nomes diferentes: 0


{}

### Correspondência por nome e equivalências adotadas

Foi decidido aceitar as equivalências abaixo para relacionar os nomes do GeoPackage aos do JSON, preservando `nmbop` e `cdbop` originais. A coluna auxiliar `nome_comparacao` incorpora essas correspondências antes da normalização de acentos, caixa e espaços.

| Nome no GeoPackage | Nome no JSON |
| --- | --- |
| CEL APARICIO BORGES | Coronel Aparício Borges |
| MONTSERRAT | Mont'Serrat |
| PASSO DAREIA | Passo D'Areia |
| SAO JOSE | Vila São José |

A correspondência de São José foi aceita com apoio na posição do polígono, que faz divisa com Partenon, e nos atributos que o associam às regiões PARTENON e LOMBA DO PINHEIRO – PARTENON. Isso não comprova igualdade dos limites territoriais adotados pelas duas fontes.

`left_only` indica nomes apenas no GeoPackage; `right_only`, apenas no JSON. Bairros restantes sem correspondência são mantidos para investigação, sem associação automática ou exclusão. Nenhum dado original é modificado e nenhuma tabela preparada é exportada nesta etapa.


In [6]:
import unicodedata


def normalizar_nome(nome):
    nome = unicodedata.normalize("NFKD", nome)
    nome = "".join(
        caractere
        for caractere in nome
        if not unicodedata.combining(caractere)
    )

    return " ".join(nome.casefold().split())


equivalencias_bairros = {
    "CEL APARICIO BORGES": "Coronel Aparício Borges",
    "MONTSERRAT": "Mont'Serrat",
    "PASSO DAREIA": "Passo D'Areia",
    "SAO JOSE": "Vila São José",
}

comparacao_gpkg = bairros[["nmbop", "cdbop"]].copy()
comparacao_gpkg["nome_comparacao"] = (
    comparacao_gpkg["nmbop"].replace(equivalencias_bairros)
)
comparacao_gpkg["chave_nome"] = (
    comparacao_gpkg["nome_comparacao"].map(
        lambda nome: normalizar_nome(nome) if pd.notna(nome) else None
    )
)

comparacao_json = pd.DataFrame({
    "bairro_json": sorted(nomes_json)
})
comparacao_json["chave_nome"] = (
    comparacao_json["bairro_json"].map(normalizar_nome)
)

comparacao = comparacao_gpkg.merge(
    comparacao_json,
    on="chave_nome",
    how="outer",
    indicator=True
)

print("Resumo da correspondência:")
display(comparacao["_merge"].value_counts())

print("\nNomes com chaves repetidas após normalização:")
display(comparacao_gpkg.loc[
    comparacao_gpkg["chave_nome"].duplicated(keep=False)
])
display(comparacao_json.loc[
    comparacao_json["chave_nome"].duplicated(keep=False)
])

print("\nBairros sem correspondência por nome:")
display(comparacao.loc[
    comparacao["_merge"] != "both",
    ["nmbop", "cdbop", "bairro_json", "_merge"]
])

Resumo da correspondência:


_merge
both          83
right_only    11
left_only      0
Name: count, dtype: int64


Nomes com chaves repetidas após normalização:


,nmbop,cdbop,nome_comparacao,chave_nome


,bairro_json,chave_nome



Bairros sem correspondência por nome:


,nmbop,cdbop,bairro_json,_merge
0,NaN,NaN,Aberta dos Morros,right_only
10,NaN,NaN,Boa Vista do Sul,right_only
22,NaN,NaN,Costa e Silva,right_only
26,NaN,NaN,Extrema,right_only
40,NaN,NaN,Jardim Europa,right_only
44,NaN,NaN,Jardim Leopoldina,right_only
59,NaN,NaN,Parque Santa Fé,right_only
65,NaN,NaN,Pitinga,right_only
73,NaN,NaN,Santa Rosa de Lima,right_only
77,NaN,NaN,São Caetano,right_only


### Seleção dos atributos dos bairros

`bairros_selecionados` é uma cópia geoespacial que mantém os 83 registros, o CRS e as geometrias originais.

| Atributo | Finalidade |
| --- | --- |
| `nmbop` | Nome original do bairro. |
| `cdbop` | Código territorial da fonte. |
| `bairro_json` | Nome correspondente no JSON, considerando as quatro equivalências adotadas. |
| `cota_mean` | Atributo preservado para possível uso futuro; provável elevação média, com unidade, referência vertical e metodologia ainda não confirmadas. Não é usado para calcular profundidade ou classificar inundação. |
| `geometry` | Geometria para possíveis mapas e cruzamentos espaciais. |

Os campos de regiões administrativas e de planejamento ficam fora desta seleção inicial e permanecem disponíveis na camada original. Os 11 bairros exclusivos do JSON continuam sinalizados na comparação anterior, sem criação de geometrias ou exclusão de seus indicadores.

As verificações abaixo conferem a correspondência de um para um e a preservação dos registros, do CRS e das geometrias. A exportação e a conferência do arquivo estão na etapa seguinte.


In [7]:
correspondencias = comparacao.loc[
    comparacao["_merge"] == "both",
    ["cdbop", "bairro_json"]
].copy()

assert correspondencias["cdbop"].is_unique, "Há códigos com correspondência ambígua."
assert correspondencias["bairro_json"].is_unique, "Há nomes do JSON com correspondência ambígua."

bairros_selecionados = bairros[
    ["nmbop", "cdbop", "cota_mean", "geometry"]
].copy()

bairros_selecionados["bairro_json"] = bairros_selecionados["cdbop"].map(
    correspondencias.set_index("cdbop")["bairro_json"]
)

bairros_selecionados = bairros_selecionados[
    ["nmbop", "cdbop", "bairro_json", "cota_mean", "geometry"]
]

assert len(bairros_selecionados) == len(bairros), "A quantidade de bairros foi alterada."
assert bairros_selecionados["cdbop"].is_unique, "Há códigos territoriais repetidos."
assert bairros_selecionados["bairro_json"].notna().all(), "Há bairros sem correspondência no JSON."
assert bairros_selecionados.crs == bairros.crs, "O CRS foi alterado."
assert bairros_selecionados.geometry.equals(bairros.geometry), "As geometrias foram alteradas."

print(f"Bairros selecionados: {len(bairros_selecionados)}")
print(f"Atributos: {list(bairros_selecionados.columns)}")
print(f"CRS preservado: {bairros_selecionados.crs}")
display(bairros_selecionados)

Bairros selecionados: 83
Atributos: ['nmbop', 'cdbop', 'bairro_json', 'cota_mean', 'geometry']
CRS preservado: EPSG:31982


,nmbop,cdbop,bairro_json,cota_mean,geometry
0,AGRONOMIA,BOP71,Agronomia,125.157306,"MULTIPOLYGON Z (((489305.255 6676873.088 0, 48..."
1,ANCHIETA,BOP19,Anchieta,1.954820,"MULTIPOLYGON Z (((484961.018 6685092.233 0, 48..."
2,ARQUIPELAGO,BOP36,Arquipélago,1.162602,"MULTIPOLYGON Z (((471983.597 6674237.441 0, 47..."
3,AUXILIADORA,BOP01,Auxiliadora,28.703022,"MULTIPOLYGON Z (((482207 6678761.422 0, 482243..."
4,AZENHA,BOP02,Azenha,12.897710,"MULTIPOLYGON Z (((479842.718 6675588.541 0, 47..."
...,...,...,...,...,...
78,SANTA MARIA GORETTI,BOP32,Santa Maria Goretti,4.145914,"MULTIPOLYGON Z (((483675.3 6680321.662 0, 4836..."
79,CAVALHADA,BOP57,Cavalhada,23.222729,"MULTIPOLYGON Z (((478348.363 6670374.305 0, 47..."
80,PRAIA DE BELAS,BOP15,Praia de Belas,2.401748,"MULTIPOLYGON Z (((477146.457 6677062.151 0, 47..."
81,PONTA GROSSA,BOP83,Ponta Grossa,14.849819,"MULTIPOLYGON Z (((482726.295 6662157.991 0, 48..."


### Exportação e conferência dos bairros preparados

Os cinco atributos selecionados são salvos em `dados/tratados/poa_admin_bairros.parquet`. A releitura compara registros, atributos, tipos, CRS e geometrias com `bairros_selecionados`.

O arquivo preserva `cota_mean` com interpretação pendente e as equivalências de nomes adotadas; não inclui geometrias para os 11 bairros exclusivos do JSON. O GeoPackage original permanece preservado.


In [8]:
DADOS_TRATADOS.mkdir(parents=True, exist_ok=True)
ARQUIVO_BAIRROS_TRATADOS = DADOS_TRATADOS / "poa_admin_bairros.parquet"

bairros_selecionados.to_parquet(ARQUIVO_BAIRROS_TRATADOS, index=False)

bairros_conferidos = gpd.read_parquet(ARQUIVO_BAIRROS_TRATADOS)

pd.testing.assert_frame_equal(bairros_conferidos, bairros_selecionados)
assert bairros_conferidos.crs == bairros_selecionados.crs, "O CRS não foi preservado."
assert bairros_conferidos.geometry.equals(bairros_selecionados.geometry), "As geometrias não foram preservadas."

print(f"Registros conferidos: {len(bairros_conferidos)}")
print("Atributos, tipos, valores, CRS e geometrias preservados na releitura.")

Registros conferidos: 83
Atributos, tipos, valores, CRS e geometrias preservados na releitura.


## 3. Educação — localização e identificação das escolas

**Camada:** `poa_edu_escolas`.

Foi decidido selecionar os atributos abaixo para identificação, agrupamentos e possíveis mapas e cruzamentos espaciais. Matrículas e indicadores educacionais ficam fora desta seleção.

| Atributo | Uso |
| --- | --- |
| `INEP` | Identificação da escola. |
| `Name` | Nome original. |
| `C__digo` | Código adicional da fonte. |
| `BAIRRO_GEO` | Bairro informado. |
| `Redes_de_E` | Rede de ensino. |
| `geometry` | Localização geográfica. |

A seleção é feita em uma cópia, sem renomear colunas, corrigir coordenadas ou remover registros. Permanece o critério de considerar os 1.062 registros pertencentes a Porto Alegre, preservando os cinco pontos externos à cobertura dos polígonos de bairros. Coordenadas compartilhadas não são removidas automaticamente. A seleção não confirma a validade externa dos códigos INEP ou a precisão das localizações.


In [9]:
escolas = gpd.read_file(
    ARQUIVO_BANCO,
    layer="poa_edu_escolas"
)

atributos_escolas = [
    "INEP",
    "Name",
    "C__digo",
    "BAIRRO_GEO",
    "Redes_de_E",
    "geometry",
]

escolas_selecionadas = escolas[atributos_escolas].copy()

assert len(escolas_selecionadas) == len(escolas), "A quantidade de registros foi alterada."
assert escolas_selecionadas.crs == escolas.crs, "O CRS foi alterado."
assert escolas_selecionadas.geometry.equals(escolas.geometry), "As geometrias foram alteradas."
pd.testing.assert_frame_equal(escolas_selecionadas, escolas[atributos_escolas])

print(f"Registros preservados: {len(escolas_selecionadas)}")
print(f"CRS: {escolas_selecionadas.crs}")
print("Ausências nos atributos selecionados:")
display(escolas_selecionadas.isna().sum())

with pd.option_context("display.max_rows", 100, "display.max_columns", None):
    display(escolas_selecionadas.head(100))

Registros preservados: 1062
CRS: EPSG:4326
Ausências nos atributos selecionados:


INEP          0
Name          0
C__digo       0
BAIRRO_GEO    0
Redes_de_E    0
geometry      0
dtype: int64

,INEP,Name,C__digo,BAIRRO_GEO,Redes_de_E,geometry
0,43187137,5066 EEI MEU NENE (C PE ),258P,AGRONOMIA,PARTICULAR,POINT Z (-51.12672 -30.08765 0)
1,43105769,EEEM AGRONOMO PEDRO PEREIRA ( EF EM EJA),21E,AGRONOMIA,ESTADUAL,POINT Z (-51.13402 -30.07458 0)
2,43108776,EEEF IRMAO MIGUEL DARIO ( EJA),220E,AGRONOMIA,ESTADUAL,POINT Z (-51.14153 -30.06961 0)
3,43104932,COLEGIO DE APLICACAO UFRGS ( EF EM EJA),2F,AGRONOMIA,FEDERAL,POINT Z (-51.1257 -30.07602 0)
4,43106153,EEEF DESIDERIO TORQUATO FINAMOR ( EF ),57E,AGRONOMIA,ESTADUAL,POINT Z (-51.14153 -30.06961 0)
5,43367380,5198 EEI MEUS PRIMEIROS PASSOS (C PE ),676P,AGRONOMIA,PARTICULAR,POINT Z (-51.13132 -30.07624 0)
6,43108725,EEEF ONOFRE PIRES ( EF ),215E,AGRONOMIA,ESTADUAL,POINT Z (-51.13929 -30.08423 0)
7,43108849,ESCOLA FUNDAMENTAL LA SALLE ESMERALDA ( EF ),164P,AGRONOMIA,PARTICULAR,POINT Z (-51.12532 -30.08874 0)
8,43107966,EEEF PROFESSOR SYLVIO TORRES ( EF EJA),180E,AGRONOMIA,ESTADUAL,POINT Z (-51.11614 -30.0807 0)
9,43297110,5185 EEI SAO MARCOS (C PE ),636P,AGRONOMIA,PARTICULAR,POINT Z (-51.1157 -30.08407 0)


### Outros possíveis caminhos de análise das escolas

| Possível análise | Pergunta ou contribuição | Condições e dados necessários |
| --- | --- | --- |
| Exposição por bairro | Quantas escolas e qual proporção delas intersectam as manchas em cada bairro? | Validar manchas e correspondência territorial; definir como tratar os cinco pontos externos à cobertura dos bairros, sem excluí-los automaticamente. |
| Exposição por rede de ensino | Como a exposição se distribui entre redes municipais, estaduais, federais e particulares? | Conferir as categorias de `Redes_de_E` e considerar o total de escolas de cada rede. |
| Comparação entre cenários | Como poderia mudar a exposição das escolas entre os cenários já disponíveis no INCT? | Depende de disponibilidade das manchas ou outra representação espacial dos cenários do INCT e de compatibilidade metodológica. O JSON agregado por bairro, isoladamente, não localiza a inundação em cada escola. As simulações deste GeoPackage ficam fora da preparação. |
| Distribuição de escolas fora das manchas | Existem estabelecimentos próximos às áreas expostas que não intersectam a mancha escolhida? | A localização pode orientar uma investigação; acolhimento ou transferência de alunos exigiriam capacidade, acesso, funcionamento e compatibilidade das etapas de ensino. Estar fora da mancha não comprova segurança ou disponibilidade. |
| Exposição conjunta de educação e saúde | Quais bairros concentram escolas e serviços de saúde espacialmente expostos? | Preparar as camadas de saúde, esclarecer a unidade de contagem dos serviços e evitar dupla contagem entre cadastros. |

Uma possibilidade inicial é avaliar exposição por bairro e rede, apresentando **quantidades e proporções** com denominadores e cobertura explícitos. Pontos com bairro informado, mas fora dos polígonos, podem exigir um critério documentado de agrupamento.

Esses caminhos permanecem propostas, sem cruzamentos realizados nesta etapa. A interseção de uma escola com uma mancha indica **exposição espacial**, não comprovação de danos, interrupção de aulas ou efeitos educacionais. A referência temporal do cadastro escolar também precisa ser considerada.


### Possível pesquisa complementar — desempenho no ENEM 2024

Foi definido como próximo caminho de pesquisa investigar a disponibilidade de dados de desempenho no **ENEM de 2024**, incluindo a possibilidade de relacioná-los às escolas pelo código INEP. Essa pesquisa ainda não foi realizada; a disponibilidade, a identificação da escola e a granularidade dos resultados não estão confirmadas.

Os seis atributos selecionados preservam identificação e localização para essa investigação. Um eventual estudo precisaria confirmar o funcionamento, o endereço e a oferta de ensino médio em 2024, além da compatibilidade das coordenadas com a data da enchente. Resultados de 2024, isoladamente, não demonstram efeito da inundação: seriam necessárias referências anteriores, grupos comparáveis e avaliação de mudanças na participação no exame.


### Exportação e conferência das escolas

Os seis atributos selecionados são salvos em `dados/tratados/poa_edu_escolas.parquet`. A releitura confere registros, colunas, tipos, valores, CRS e geometrias. A exportação preserva as 1.062 escolas, inclusive coordenadas compartilhadas e os cinco pontos externos à cobertura territorial; não constitui confirmação da existência ou localização de cada estabelecimento em 2024.


In [10]:
DADOS_TRATADOS.mkdir(parents=True, exist_ok=True)
ARQUIVO_ESCOLAS_TRATADAS = DADOS_TRATADOS / "poa_edu_escolas.parquet"

escolas_selecionadas.to_parquet(ARQUIVO_ESCOLAS_TRATADAS, index=False)

escolas_conferidas = gpd.read_parquet(ARQUIVO_ESCOLAS_TRATADAS)

pd.testing.assert_frame_equal(escolas_conferidas, escolas_selecionadas)
assert escolas_conferidas.crs == escolas_selecionadas.crs, "O CRS não foi preservado."
assert escolas_conferidas.geometry.equals(escolas_selecionadas.geometry), "As geometrias não foram preservadas."

print(f"Registros conferidos: {len(escolas_conferidas)}")
print("Atributos, tipos, valores, CRS e geometrias preservados na releitura.")

Registros conferidos: 1062
Atributos, tipos, valores, CRS e geometrias preservados na releitura.


## 4. Saúde — cadastro SUS

**Camada:** `poa_saude_sus`. Mantemos `id01`, `Name`, `classe` e `geometry`, sem exclusão de atributos. Esta seção separa cada diagnóstico e sua saída. As tabelas grandes são apresentadas como prévias; os cálculos e as variáveis preservam todos os registros.


### 4.1 Leitura dos registros

A prévia mostra dez registros do cadastro. `id01` identifica uma linha na fonte, `Name` é o nome informado e `classe` é a categoria do serviço.


In [11]:
saude_sus = gpd.read_file(ARQUIVO_BANCO, layer="poa_saude_sus")
print(f"Registros: {len(saude_sus)} | CRS: {saude_sus.crs}")
with pd.option_context("display.max_columns", None):
    display(saude_sus.head(10))

Registros: 353 | CRS: EPSG:4326


,Name,classe,id01,geometry
0,CLINICAS,hospital,1,POINT Z (-51.20804 -30.03912 0)
1,PRONTO SOCORRO,hospital,2,POINT Z (-51.2096 -30.03557 0)
2,COMPLEXO SANTA CASA,hospital,3,POINT Z (-51.21986 -30.0288 0)
3,HMIPV,hospital,4,POINT Z (-51.21467 -30.02863 0)
4,HOSPITAL BENEFICENCIA PORTUGUESA,hospital,5,POINT Z (-51.21885 -30.02861 0)
5,FEMINA,hospital,6,POINT Z (-51.20725 -30.02819 0)
6,MOINHOS DE VENTO,hospital,7,POINT Z (-51.20905 -30.02627 0)
7,CENTRO DE DIÁLISE E TRANSPLANTE,hospital,8,POINT Z (-51.20353 -30.01854 0)
8,PETROPOLIS,hospital,9,POINT Z (-51.19432 -30.03972 0)
9,ULBRA,hospital,10,POINT Z (-51.19794 -30.03657 0)


### 4.2 Tipos e preenchimento

Uma única tabela reúne o tipo de cada coluna, as ausências e os textos vazios. Texto vazio é verificado apenas nos campos de nome e categoria.


In [12]:
diagnostico_atributos_sus = pd.DataFrame({
    "tipo": saude_sus.dtypes.astype(str),
    "ausentes": saude_sus.isna().sum(),
})
diagnostico_atributos_sus["textos_vazios"] = pd.Series(dtype="Int64")
for coluna in ["Name", "classe"]:
    diagnostico_atributos_sus.loc[coluna, "textos_vazios"] = (
        saude_sus[coluna].astype("string").str.strip().eq("").fillna(False).sum()
    )
display(diagnostico_atributos_sus)

,tipo,ausentes,textos_vazios
Name,str,0,0
classe,str,0,0
id01,int64,0,<NA>
geometry,geometry,0,<NA>


### 4.3 Identificadores e linhas repetidas

Contamos registros com `id01` repetido e linhas inteiramente duplicadas. Eventuais casos aparecem em uma prévia de até dez linhas; não são removidos.


In [13]:
sus_ids_repetidos = saude_sus.loc[saude_sus["id01"].duplicated(keep=False)]
print(f"Registros com id01 repetido: {len(sus_ids_repetidos)}")
print(f"Linhas inteiramente duplicadas: {saude_sus.duplicated().sum()}")
if not sus_ids_repetidos.empty:
    display(sus_ids_repetidos.head(10))

Registros com id01 repetido: 0
Linhas inteiramente duplicadas: 0


### 4.4 Categorias de serviços

A tabela conta registros por categoria original. Esses totais não equivalem necessariamente a prédios ou estabelecimentos distintos.


In [14]:
display(saude_sus["classe"].value_counts(dropna=False).rename_axis("classe").reset_index(name="registros"))

,classe,registros
0,ubs,106
1,usf,89
2,caps,34
3,hospital,28
4,base samu,28
5,cs,14
6,saude mental,11
7,upa,10
8,farmacia distrital,9
9,gerencia distrital,8


### 4.5 Qualidade das geometrias

Verificamos geometrias nulas, vazias e inválidas. A validade não comprova precisão da localização ou funcionamento do serviço em 2024.


In [15]:
sus_geometria_nula = saude_sus.geometry.isna()
sus_geometria_vazia = saude_sus.geometry.is_empty
sus_geometria_invalida = ~sus_geometria_nula & ~sus_geometria_vazia & ~saude_sus.geometry.is_valid
sus_geometria_utilizavel = ~sus_geometria_nula & ~sus_geometria_vazia & ~sus_geometria_invalida

display(pd.Series({
    "nulas": int(sus_geometria_nula.sum()),
    "vazias": int(sus_geometria_vazia.sum()),
    "inválidas não vazias": int(sus_geometria_invalida.sum()),
    "utilizáveis": int(sus_geometria_utilizavel.sum()),
}, name="registros").to_frame())
if (~sus_geometria_utilizavel).any():
    display(saude_sus.loc[~sus_geometria_utilizavel].head(10))

,registros
nulas,0
vazias,0
inválidas não vazias,0
utilizáveis,353


### 4.6 Quantificação das posições compartilhadas

Comparamos longitude (X) e latitude (Y) por igualdade exata, sem arredondamento e desconsiderando Z. Registros muito próximos, mas com coordenadas diferentes, não são agrupados. Esta saída apresenta apenas os totais; não modifica os pontos.


In [16]:
sus_pontos_utilizaveis = sus_geometria_utilizavel & saude_sus.geom_type.eq("Point")
saude_sus_localizacoes = saude_sus.loc[sus_pontos_utilizaveis].copy()
saude_sus_localizacoes["x_original"] = saude_sus_localizacoes.geometry.x
saude_sus_localizacoes["y_original"] = saude_sus_localizacoes.geometry.y

sus_coordenadas_repetidas = saude_sus_localizacoes.duplicated(
    subset=["x_original", "y_original"], keep=False
)
saude_sus_coincidentes = saude_sus_localizacoes.loc[
    sus_coordenadas_repetidas
].sort_values(["x_original", "y_original", "classe", "Name"])

resumo_locais_sus = (
    saude_sus_coincidentes.groupby(["x_original", "y_original"], as_index=False)
    .agg(
        registros=("id01", "size"),
        nomes_distintos=("Name", "nunique"),
        categorias_distintas=("classe", "nunique"),
    )
    .sort_values("registros", ascending=False)
)

print(f"Registros com coordenadas compartilhadas: {len(saude_sus_coincidentes)}")
print(f"Locais compartilhados: {len(resumo_locais_sus)}")
print(f"Posições horizontais distintas: {saude_sus_localizacoes[['x_original', 'y_original']].drop_duplicates().shape[0]}")
print(f"Registros fora da comparação: {len(saude_sus) - len(saude_sus_localizacoes)}")

Registros com coordenadas compartilhadas: 223
Locais compartilhados: 89
Posições horizontais distintas: 219
Registros fora da comparação: 0


### 4.7 Locais com maior concentração de registros

A prévia apresenta os **dez locais com mais registros**, uma linha por posição. `registros` conta as linhas do cadastro; `nomes_distintos` e `categorias_distintas` contam os valores diferentes naquele ponto. `local_sus` é um identificador auxiliar, sem caráter oficial.

Os nomes e serviços não são listados nesta tabela para evitar células extensas; o próximo bloco detalha um local por vez. Todos os locais permanecem em `locais_sus_detalhados`.


In [17]:
locais_sus_detalhados = (
    saude_sus_coincidentes.groupby(["x_original", "y_original"], as_index=False)
    .agg(
        registros=("id01", "size"),
        nomes_distintos=("Name", "nunique"),
        categorias_distintas=("classe", "nunique"),
        nomes=("Name", lambda valores: sorted(valores.unique().tolist())),
        categorias=("classe", lambda valores: sorted(valores.unique().tolist())),
        ids_registros=("id01", lambda valores: sorted(valores.tolist())),
    )
    .sort_values(
        ["registros", "x_original", "y_original"],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)
locais_sus_detalhados.insert(0, "local_sus", range(1, len(locais_sus_detalhados) + 1))

registros_locais_sus = saude_sus_coincidentes.merge(
    locais_sus_detalhados[["local_sus", "x_original", "y_original"]],
    on=["x_original", "y_original"],
    how="left",
    validate="many_to_one"
).sort_values(["local_sus", "classe", "Name", "id01"])

registros_locais_sus["nome_categoria_repetidos_no_local"] = (
    registros_locais_sus.duplicated(
        subset=["local_sus", "Name", "classe"], keep=False
    )
)

with pd.option_context("display.max_columns", None):
    display(locais_sus_detalhados[
        ["local_sus", "x_original", "y_original", "registros", "nomes_distintos", "categorias_distintas"]
    ].head(10))

,local_sus,x_original,y_original,registros,nomes_distintos,categorias_distintas
0,1,-51.232379,-30.027663,13,10,9
1,2,-51.216588,-30.069445,11,5,7
2,3,-51.235456,-30.107524,6,4,5
3,4,-51.178469,-30.062400,6,4,5
4,5,-51.214118,-30.042497,5,3,3
5,6,-51.178238,-30.013622,4,3,3
6,7,-51.163665,-30.045253,4,2,3
7,8,-51.140488,-30.157804,4,3,3
8,9,-51.128258,-30.038080,4,3,3
9,10,-51.126661,-30.015982,4,3,3


### 4.8 Registros de um local compartilhado

Esta tabela detalha **um único local** do resumo anterior. Cada linha é um serviço cadastrado naquela coordenada, com nome, categoria e identificador. A marcação `nome_categoria_repetidos_no_local` sinaliza a repetição exata desse par no ponto, sem concluir que haja duplicação indevida.

O parâmetro `LOCAL_SUS_ANALISADO` define o local apresentado. O valor inicial 1 corresponde à posição com mais registros.


In [18]:
LOCAL_SUS_ANALISADO = 1
registros_local_analisado = registros_locais_sus.loc[
    registros_locais_sus["local_sus"] == LOCAL_SUS_ANALISADO,
    ["id01", "Name", "classe", "nome_categoria_repetidos_no_local"]
]
print(f"Local {LOCAL_SUS_ANALISADO}: {len(registros_local_analisado)} registros")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 60):
    display(registros_local_analisado.reset_index(drop=True))

Local 1: 13 registros


,id01,Name,classe,nome_categoria_repetidos_no_local
0,155,CEO CENTRO,ceo,True
1,344,CEO CENTRO,ceo,True
2,148,CONSULTÓRIO NA RUA CENTRO,consultorio na rua,False
3,157,CS SANTA MARTA,cs,True
4,347,CS SANTA MARTA,cs,True
5,153,CENTRO,eesca,False
6,154,SANTA MARTA,farmacia distrital,False
7,149,GD CENTRO,gerencia distrital,False
8,152,AMBULATÓRIO SAÚDE MENTAL ADULTO CENTRO,saude mental,False
9,48,UBS SANTA MARTA,ubs,True


### 4.9 Mesmo nome e categoria no mesmo ponto

Cada linha abaixo representa um grupo de registros com nome e categoria idênticos em um local. Mostramos os dez grupos com mais repetições. A tabela completa permanece em `repeticoes_nome_categoria_sus`; esses casos exigem investigação antes de qualquer exclusão.


In [19]:
repeticoes_nome_categoria_sus = (
    registros_locais_sus.groupby(["local_sus", "Name", "classe"], as_index=False)
    .agg(registros=("id01", "size"))
    .query("registros > 1")
    .sort_values(["registros", "local_sus"], ascending=[False, True])
)

posicoes_por_nome_sus = (
    saude_sus_localizacoes[["Name", "x_original", "y_original"]]
    .drop_duplicates()
    .groupby("Name", as_index=False)
    .size()
    .rename(columns={"size": "posicoes_distintas"})
    .query("posicoes_distintas > 1")
    .sort_values(["posicoes_distintas", "Name"], ascending=[False, True])
)

nomes_em_varias_posicoes_sus = saude_sus_localizacoes.loc[
    saude_sus_localizacoes["Name"].isin(posicoes_por_nome_sus["Name"]),
    ["id01", "Name", "classe", "x_original", "y_original"]
].sort_values(["Name", "x_original", "y_original", "classe"])

print(f"Grupos com mesmo nome e categoria no mesmo local: {len(repeticoes_nome_categoria_sus)}")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 60):
    display(repeticoes_nome_categoria_sus.head(10).reset_index(drop=True))

Grupos com mesmo nome e categoria no mesmo local: 67


,local_sus,Name,classe,registros
0,1,CEO CENTRO,ceo,2
1,1,CS SANTA MARTA,cs,2
2,1,UBS SANTA MARTA,ubs,2
3,2,CEO GCC,ceo,2
4,2,CRUZEIRO DO SUL,upa,2
5,2,CS VILA COMERCIARIOS,cs,2
6,2,GCC,caps,2
7,3,UBS CAMAQUA,ubs,2
8,4,CS MURIALDO,cs,2
9,5,CS MODELO,cs,2


### 4.10 Nomes encontrados em posições diferentes

Esta tabela conta quantas posições distintas estão associadas ao mesmo nome original. Mostramos até 20 nomes. Nomes genéricos podem designar serviços diferentes, e variações de escrita não são reunidas automaticamente.


In [20]:
print(f"Nomes presentes em mais de uma posição: {len(posicoes_por_nome_sus)}")
display(posicoes_por_nome_sus.head(20).reset_index(drop=True))

Nomes presentes em mais de uma posição: 20


,Name,posicoes_distintas
0,RESTINGA,4
1,BOM JESUS,3
2,CRISTO REDENTOR,3
3,CRUZEIRO DO SUL,3
4,LOMBA DO PINHEIRO,3
5,NAVEGANTES,3
6,BELÉM NOVO,2
7,CAVALHADA,2
8,CENTRO,2
9,CENTRO VIDA,2


### 4.11 Conferência de um nome em várias posições

A saída detalha um nome da tabela anterior. O parâmetro `NOME_SUS_ANALISADO` define o nome apresentado; todas as ocorrências continuam disponíveis em `nomes_em_varias_posicoes_sus`.


In [21]:
NOME_SUS_ANALISADO = (
    posicoes_por_nome_sus.iloc[0]["Name"] if not posicoes_por_nome_sus.empty else None
)
registros_nome_analisado = nomes_em_varias_posicoes_sus.loc[
    nomes_em_varias_posicoes_sus["Name"] == NOME_SUS_ANALISADO
]
print(f"Nome analisado: {NOME_SUS_ANALISADO}")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 60):
    display(registros_nome_analisado.reset_index(drop=True))

Nome analisado: RESTINGA


,id01,Name,classe,x_original,y_original
0,343,RESTINGA,base samu,-51.145841,-30.148778
1,33,RESTINGA,upa,-51.145053,-30.148222
2,327,RESTINGA,upa,-51.145053,-30.148222
3,326,RESTINGA,base samu,-51.142877,-30.148885
4,328,RESTINGA,farmacia distrital,-51.140107,-30.146865


### Encerramento do diagnóstico SUS

Os diagnósticos anteriores utilizam a camada original. A próxima etapa aplica o critério específico de deduplicação decidido, preservando as tabelas de investigação. A referência temporal e a precisão das posições permanecem pendentes.


### 4.12 Remoção de registros repetidos no mesmo ponto

Foi decidido manter apenas a primeira ocorrência de registros com **`Name`, `classe`, X e Y exatamente iguais**. O critério desconsidera Z, não normaliza os textos e não considera apenas o nome suficiente para excluir uma linha.

Registros com o mesmo nome em posições diferentes, classes diferentes no mesmo local ou geometria não utilizável são preservados. `id01` é mantido na ocorrência retida; os identificadores removidos são apresentados para rastreabilidade. A camada original `saude_sus` e o GeoPackage não são alterados. O resultado é armazenado em `saude_sus_preparada`, sem exportação nesta etapa.


In [22]:
chaves_duplicacao_sus = ["Name", "classe", "x_original", "y_original"]

sus_ocorrencias_excedentes = saude_sus_localizacoes.duplicated(
    subset=chaves_duplicacao_sus, keep="first"
)
indices_removidos_sus = saude_sus_localizacoes.index[sus_ocorrencias_excedentes]

saude_sus_removidos = saude_sus_localizacoes.loc[
    indices_removidos_sus,
    ["id01", "Name", "classe", "x_original", "y_original"]
].copy()
saude_sus_preparada = saude_sus.drop(index=indices_removidos_sus).copy()

# Confirmar que cada removido tem uma ocorrência equivalente preservada.
sus_pontos_mantidos = saude_sus_localizacoes.drop(index=indices_removidos_sus)
conferencia_remocoes_sus = saude_sus_removidos.merge(
    sus_pontos_mantidos[chaves_duplicacao_sus + ["id01"]].rename(
        columns={"id01": "id01_preservado"}
    ),
    on=chaves_duplicacao_sus,
    how="left",
    validate="many_to_one",
    indicator=True,
)
assert conferencia_remocoes_sus["_merge"].eq("both").all()
assert not sus_pontos_mantidos.duplicated(subset=chaves_duplicacao_sus).any()
assert len(saude_sus_preparada) == len(saude_sus) - len(saude_sus_removidos)
assert saude_sus_preparada.crs == saude_sus.crs
pd.testing.assert_frame_equal(
    saude_sus_preparada,
    saude_sus.loc[~saude_sus.index.isin(indices_removidos_sus)]
)

print(f"Registros originais: {len(saude_sus)}")
print(f"Ocorrências excedentes removidas: {len(saude_sus_removidos)}")
print(f"Registros preservados: {len(saude_sus_preparada)}")
print("Critério: nome, classe e coordenadas horizontais iguais.")

Registros originais: 353
Ocorrências excedentes removidas: 67
Registros preservados: 286
Critério: nome, classe e coordenadas horizontais iguais.


### 4.13 Conferência das ocorrências removidas

A prévia lista até 20 linhas removidas e o `id01_preservado` de sua ocorrência equivalente. A classe aparece na tabela para confirmar que serviços de categorias diferentes não foram eliminados por compartilharem nome e posição. A tabela completa permanece em `conferencia_remocoes_sus`.


In [23]:
with pd.option_context("display.max_columns", None, "display.max_colwidth", 60):
    display(conferencia_remocoes_sus[
        ["id01", "id01_preservado", "Name", "classe", "x_original", "y_original"]
    ].head(20))

,id01,id01_preservado,Name,classe,x_original,y_original
0,91,29,MOACIR SCLIAR,upa,-51.146047,-30.008630
1,98,34,UBS SARANDI,ubs,-51.134075,-29.993928
2,99,35,UBS ASSIS BRASIL,ubs,-51.128661,-29.992760
3,104,36,UBS NOVA BRASILIA,ubs,-51.134447,-29.981646
4,105,37,UBS VILA ELISABETE,ubs,-51.126541,-29.979447
5,109,38,UBS RUBEM BERTA,ubs,-51.102849,-30.023575
6,111,39,UBS SAO CRISTOVAO,ubs,-51.098261,-30.013056
7,114,40,UBS SANTA ROSA,ubs,-51.099253,-29.997240
8,116,41,UBS RAMOS,ubs,-51.105597,-29.993460
9,120,42,UBS MORRO SANTANA,ubs,-51.128258,-30.038080


### 4.14 Exportação e conferência do cadastro SUS

Foi decidido preservar registros com classes ou posições diferentes, mesmo quando o nome coincide: nomes podem designar um bairro ou região, sem identificar o mesmo serviço. Foram removidas somente ocorrências excedentes com nome, classe e coordenadas X/Y exatamente iguais, mantendo a primeira.

O resultado é salvo em `dados/tratados/poa_saude_sus.parquet`, com `id01`, `Name`, `classe` e `geometry`. A releitura confere registros, tipos, valores, CRS e geometrias. Os identificadores originais não são renumerados; as ocorrências removidas permanecem rastreáveis nas tabelas anteriores. O arquivo original não é alterado.


In [24]:
DADOS_TRATADOS.mkdir(parents=True, exist_ok=True)
ARQUIVO_SUS_TRATADO = DADOS_TRATADOS / "poa_saude_sus.parquet"

saude_sus_preparada.to_parquet(ARQUIVO_SUS_TRATADO, index=False)

saude_sus_conferida = gpd.read_parquet(ARQUIVO_SUS_TRATADO)

# A exportação não armazena o índice auxiliar do DataFrame.
pd.testing.assert_frame_equal(
    saude_sus_conferida,
    saude_sus_preparada.reset_index(drop=True)
)
assert saude_sus_conferida.crs == saude_sus_preparada.crs, "O CRS não foi preservado."
assert saude_sus_conferida.geometry.equals(
    saude_sus_preparada.geometry.reset_index(drop=True)
), "As geometrias não foram preservadas."

print(f"Registros conferidos: {len(saude_sus_conferida)}")
print("Atributos, tipos, valores, CRS e geometrias preservados na releitura.")

Registros conferidos: 286
Atributos, tipos, valores, CRS e geometrias preservados na releitura.


## 5. Atenção primária — versões temporais

**Camadas:** `poa_saude_unidades_atencao_primaria_09052024` e `poa_saude_unidades_atencao_primaria_10052024`.

A seção apresenta o diagnóstico das versões originais, a comparação de nomes, posições e situações, os ajustes adotados e a exportação separada das duas datas. Os registros não representam unidades adicionais a serem somadas entre versões ou combinadas automaticamente com o cadastro SUS.


### 5.1 Leitura das versões

As duas camadas são carregadas separadamente. A tabela abaixo apresenta uma prévia de cinco unidades por data. O campo de nome é `Nome` em 09/05 e `name` em 10/05; a padronização ocorre apenas nas cópias de comparação.


In [25]:
atencao_09 = gpd.read_file(ARQUIVO_BANCO, layer="poa_saude_unidades_atencao_primaria_09052024")
atencao_10 = gpd.read_file(ARQUIVO_BANCO, layer="poa_saude_unidades_atencao_primaria_10052024")

previa_atencao = pd.concat([
    atencao_09.rename(columns={"Nome": "nome"}).head(5).assign(data_referencia="09/05/2024"),
    atencao_10.rename(columns={"name": "nome"}).head(5).assign(data_referencia="10/05/2024"),
], ignore_index=True)
with pd.option_context("display.max_columns", None, "display.max_colwidth", 50):
    display(previa_atencao[["data_referencia", "nome", "Bairro", "Cidade", "Situacao"]])

,data_referencia,nome,Bairro,Cidade,Situacao
0,09/05/2024,Clínica da Família José Mauro Ceratti Lopes,Restinga,Porto Alegre,Aberta
1,09/05/2024,Clínica da Família Álvaro Difini,Restinga,Porto Alegre,Aberta
2,09/05/2024,US Alto Erechim,Nonoai,Porto Alegre,Aberta
3,09/05/2024,US Aparício Borges,Coronel Aparício Borges,Porto Alegre,Aberta
4,09/05/2024,US Asa Branca,Sarandi,Porto Alegre,Fechada
5,10/05/2024,Clínica da Família Álvaro Difini,Restinga,Porto Alegre,Aberta
6,10/05/2024,Clínica da Família Campo da Tuca,Partenon,Porto Alegre,Aberta
7,10/05/2024,Clínica da Família José Mauro Ceratti Lopes,Restinga,Porto Alegre,Aberta
8,10/05/2024,US Alto Erechim,Nonoai,Porto Alegre,Aberta
9,10/05/2024,US Aparício Borges,Coronel Aparício Borges,Porto Alegre,Aberta


### 5.2 Diagnóstico por versão

Nos dados originais, conferimos quantidade, preenchimento, nomes repetidos e geometrias. A ausência de erros geométricos não confirma precisão da localização. Uma chave de comparação usa a normalização textual já definida para bairros, sem alterar os nomes originais.


In [26]:
atencao_comparacao_09 = atencao_09.rename(columns={"Nome": "nome"}).copy()
atencao_comparacao_10 = atencao_10.rename(columns={"name": "nome"}).copy()

diagnostico_atencao = []
for data, camada in [("09/05/2024", atencao_comparacao_09), ("10/05/2024", atencao_comparacao_10)]:
    camada["chave_nome"] = camada["nome"].map(
        lambda nome: normalizar_nome(nome) if pd.notna(nome) else None
    )
    nula = camada.geometry.isna()
    vazia = camada.geometry.is_empty
    invalida = ~nula & ~vazia & ~camada.geometry.is_valid
    diagnostico_atencao.append({
        "data": data, "registros": len(camada), "CRS": str(camada.crs),
        "ausências nos atributos": int(camada.drop(columns="geometry").isna().sum().sum()),
        "nomes com chave repetida": int(camada["chave_nome"].duplicated(keep=False).sum()),
        "linhas inteiramente duplicadas": int(camada.duplicated().sum()),
        "geometrias nulas": int(nula.sum()), "geometrias vazias": int(vazia.sum()),
        "geometrias inválidas não vazias": int(invalida.sum()),
    })
with pd.option_context("display.max_columns", None):
    display(pd.DataFrame(diagnostico_atencao))

,data,registros,CRS,ausências nos atributos,nomes com chave repetida,linhas inteiramente duplicadas,geometrias nulas,geometrias vazias,geometrias inválidas não vazias
0,09/05/2024,136,EPSG:4326,0,0,0,0,0,0
1,10/05/2024,136,EPSG:4326,0,0,0,0,0,0


### 5.3 Correspondência dos nomes

Nos dados originais, comparamos chaves de nomes únicas entre as versões. Essa correspondência textual é candidata a identificar a mesma unidade; não é um código oficial. A tabela resume presenças nas duas datas e nomes exclusivos de cada uma.


In [27]:
for camada in [atencao_comparacao_09, atencao_comparacao_10]:
    assert camada["chave_nome"].notna().all(), "Há nomes ausentes; investigar antes da comparação."
    assert camada["chave_nome"].ne("").all(), "Há nomes vazios; investigar antes da comparação."
    assert camada["chave_nome"].is_unique, "Há nomes ambíguos após normalização."

comparacao_atencao = atencao_comparacao_09.merge(
    atencao_comparacao_10,
    on="chave_nome", how="outer", suffixes=("_09", "_10"),
    indicator=True, validate="one_to_one",
)
display(comparacao_atencao["_merge"].value_counts().rename_axis("correspondência").reset_index(name="unidades"))

,correspondência,unidades
0,both,135
1,left_only,1
2,right_only,1


### 5.4 Unidades sem correspondência textual

Apresentamos até 20 nomes exclusivos de uma versão. Diferenças de nome ou cobertura não demonstram abertura ou fechamento. Uma tabela vazia indica que todas as chaves encontraram correspondência.


In [28]:
unidades_sem_correspondencia = comparacao_atencao.loc[
    comparacao_atencao["_merge"] != "both",
    ["nome_09", "nome_10", "Bairro_09", "Bairro_10", "_merge"]
]
with pd.option_context("display.max_columns", None, "display.max_colwidth", 60):
    display(unidades_sem_correspondencia.head(20))

,nome_09,nome_10,Bairro_09,Bairro_10,_merge
4,Fechada,NaN,Sarandi,NaN,left_only
120,NaN,US Sarandi,NaN,Sarandi,right_only


### 5.5 Situação informada por data

A tabela conta as categorias originais de `Situacao`, antes de qualquer ajuste, sem atribuir causas aos fechamentos. As datas são as referências nominais das camadas, não a confirmação do horário de atualização de cada registro.


In [29]:
situacao_por_data = pd.concat([
    atencao_09["Situacao"].value_counts(dropna=False).rename("09/05/2024"),
    atencao_10["Situacao"].value_counts(dropna=False).rename("10/05/2024"),
], axis=1).fillna(0).astype(int)
display(situacao_por_data.rename_axis("situação"))

,09/05/2024,10/05/2024
situação,,
Aberta,104,94
Fechada,32,42


### 5.6 Mudanças de situação entre unidades correspondentes

Cada linha representa uma unidade com a mesma chave de nome e situação diferente entre cadastros. Uma mudança no campo pode refletir atualização ou correção; não comprova transição operacional ou relação causal com a enchente.


In [30]:
atencao_correspondentes = comparacao_atencao.loc[comparacao_atencao["_merge"] == "both"].copy()
mudancas_situacao_atencao = atencao_correspondentes.loc[
    atencao_correspondentes["Situacao_09"] != atencao_correspondentes["Situacao_10"],
    ["nome_09", "nome_10", "Bairro_09", "Situacao_09", "Situacao_10"]
]
print(f"Unidades com situação diferente entre os registros: {len(mudancas_situacao_atencao)}")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 60):
    display(mudancas_situacao_atencao.head(20))

Unidades com situação diferente entre os registros: 9


,nome_09,nome_10,Bairro_09,Situacao_09,Situacao_10
45,US Herdeiros,US Herdeiros,Agronomia,Aberta,Fechada
53,US Jardim da Fapa,US Jardim da Fapa,Mário Quintana,Aberta,Fechada
84,US Panorama,US Panorama,Lomba do Pinheiro,Aberta,Fechada
91,US Pitoresca,US Pitoresca,Partenon,Aberta,Fechada
96,US Recreio da Divisa,US Recreio da Divisa,Lomba do Pinheiro,Aberta,Fechada
119,US São Vicente Mártir,US São Vicente Mártir,Camaquã,Aberta,Fechada
122,US Tijuca,US Tijuca,Morro Santana,Aberta,Fechada
125,US Viçosa,US Viçosa,Vila Viçosa,Aberta,Fechada
134,US Vila Safira,US Vila Safira,Mário Quintana,Aberta,Fechada


### 5.7 Comparação das posições

Comparamos X/Y das geometrias, desconsiderando Z. Distâncias são calculadas em EPSG:31982, em metros, apenas para pontos utilizáveis das duas versões. Diferenças mínimas podem resultar da precisão de armazenamento e não demonstram mudança de endereço. As camadas originais permanecem intactas.


In [31]:
def posicoes_atencao(camada, sufixo):
    utilizavel = (
        camada.geometry.notna() & ~camada.geometry.is_empty
        & camada.geometry.is_valid & camada.geom_type.eq("Point")
    )
    pontos = camada.loc[utilizavel].to_crs(31982)
    return pd.DataFrame({
        "chave_nome": pontos["chave_nome"],
        f"x_{sufixo}": pontos.geometry.x,
        f"y_{sufixo}": pontos.geometry.y,
    })

assert atencao_09.crs is not None and atencao_10.crs is not None
posicoes_atencao_comparadas = posicoes_atencao(atencao_comparacao_09, "09").merge(
    posicoes_atencao(atencao_comparacao_10, "10"),
    on="chave_nome", validate="one_to_one"
)
posicoes_atencao_comparadas["distancia_m"] = (
    (posicoes_atencao_comparadas["x_10"] - posicoes_atencao_comparadas["x_09"]) ** 2
    + (posicoes_atencao_comparadas["y_10"] - posicoes_atencao_comparadas["y_09"]) ** 2
) ** 0.5
posicoes_atencao_comparadas = posicoes_atencao_comparadas.merge(
    atencao_comparacao_09[["chave_nome", "nome"]], on="chave_nome", validate="one_to_one"
).sort_values("distancia_m", ascending=False)
print(f"Unidades com posições comparáveis: {len(posicoes_atencao_comparadas)}")
print(f"Correspondências sem duas posições utilizáveis: {len(atencao_correspondentes) - len(posicoes_atencao_comparadas)}")
print("Dez maiores distâncias entre posições registradas:")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 60):
    display(posicoes_atencao_comparadas[["nome", "distancia_m"]].head(10))

Unidades com posições comparáveis: 135
Correspondências sem duas posições utilizáveis: 0
Dez maiores distâncias entre posições registradas:


,nome,distancia_m
0,Clínica da Família José Mauro Ceratti Lopes,0.0
1,Clínica da Família Álvaro Difini,0.0
2,US Alto Erechim,0.0
3,US Aparício Borges,0.0
4,US Asa Branca,0.0
5,US Assis Brasil,0.0
6,US Bananeiras,0.0
7,US Barão de Bagé,0.0
8,US Batista Flores,0.0
9,US Beco do Adelar,0.0


### 5.8 Inconsistências de município nos dados originais

Identificamos valores de `Cidade` diferentes de Porto Alegre, preservando os valores da fonte. A tabela apresenta os nomes, bairros, situações e endereços compostos para investigar a inconsistência antes de definir ajustes.


In [32]:
inconsistencias_cidade_atencao = []
for data, camada in [("09/05/2024", atencao_comparacao_09), ("10/05/2024", atencao_comparacao_10)]:
    cidade_comparacao = camada["Cidade"].map(
        lambda valor: normalizar_nome(valor) if pd.notna(valor) else None
    )
    casos = camada.loc[
        cidade_comparacao.ne("porto alegre"),
        ["nome", "Bairro", "Cidade", "Situacao", "End_cidade"]
    ].copy()
    casos.insert(0, "data_referencia", data)
    inconsistencias_cidade_atencao.append(casos)

with pd.option_context("display.max_columns", None, "display.max_colwidth", 60):
    display(pd.concat(inconsistencias_cidade_atencao, ignore_index=True).head(20))

,data_referencia,nome,Bairro,Cidade,Situacao,End_cidade
0,09/05/2024,US Nossa Senhora Aparecida,Sarandi,Fechada,Aberta,"Beco José Paris, 235, Sarandi, Fechada"
1,10/05/2024,US Nossa Senhora Aparecida,Sarandi,Fechada,Aberta,"Beco José Paris, 235, Sarandi, Fechada"


### 5.9 Decisões após o diagnóstico

Foi decidido reconhecer como **US Sarandi**, no bairro **Sarandi**, o registro cujo nome em 09/05 estava preenchido como “Fechada”. A versão de 10/05 já apresenta “US Sarandi”. Os valores originais e as situações adotadas como **“Fechada”** constam no registro de alterações.

Para **US Nossa Senhora Aparecida**, foi decidido ajustar `Cidade` para **Porto Alegre**, reconstruir `End_cidade` com o endereço, bairro e município e adotar `Situacao = "Fechada"` nas duas versões. Essa situação é uma decisão baseada na hipótese de fechamento em decorrência do impacto das enchentes no Sarandi, **não uma confirmação documental do funcionamento da unidade**.

As alterações são restritas a esses registros, em cópias `atencao_09_preparada` e `atencao_10_preparada`. As camadas originais permanecem em `atencao_09` e `atencao_10`; a tabela de alterações preserva os valores anteriores e novos. Os demais serviços do Sarandi não são classificados automaticamente como fechados.


In [33]:
atencao_09_preparada = atencao_09.copy()
atencao_10_preparada = atencao_10.copy()

alteracoes_atencao = []
for data, camada, campo_nome in [
    ("09/05/2024", atencao_09_preparada, "Nome"),
    ("10/05/2024", atencao_10_preparada, "name"),
]:
    nome_sarandi = "Fechada" if data == "09/05/2024" else "US Sarandi"
    mascara_sarandi = (
        camada[campo_nome].str.strip().eq(nome_sarandi)
        & camada["Bairro"].str.strip().eq("Sarandi")
    )
    mascara_aparecida = camada[campo_nome].str.strip().eq("US Nossa Senhora Aparecida")
    assert mascara_sarandi.sum() == 1, "Conferir o registro correspondente à US Sarandi."
    assert mascara_aparecida.sum() == 1, "Conferir o registro da US Nossa Senhora Aparecida."

    for mascara, nome_padronizado in [
        (mascara_sarandi, "US Sarandi"),
        (mascara_aparecida, "US Nossa Senhora Aparecida"),
    ]:
        indice = camada.index[mascara][0]
        valores_novos = {
            campo_nome: nome_padronizado,
            "Bairro": "Sarandi",
            "Cidade": "Porto Alegre",
            "Situacao": "Fechada",
            "End_cidade": f"{camada.at[indice, 'End']}, Sarandi, Porto Alegre",
        }
        for campo, novo in valores_novos.items():
            anterior = camada.at[indice, campo]
            if anterior != novo:
                alteracoes_atencao.append({
                    "data": data, "unidade": nome_padronizado,
                    "campo": campo, "valor_original": anterior, "valor_adotado": novo,
                })
            camada.at[indice, campo] = novo

registro_alteracoes_atencao = pd.DataFrame(alteracoes_atencao)
assert len(atencao_09_preparada) == len(atencao_09)
assert len(atencao_10_preparada) == len(atencao_10)
assert atencao_09_preparada.geometry.equals(atencao_09.geometry)
assert atencao_10_preparada.geometry.equals(atencao_10.geometry)

with pd.option_context("display.max_columns", None, "display.max_colwidth", 65):
    display(registro_alteracoes_atencao)

,data,unidade,campo,valor_original,valor_adotado
0,09/05/2024,US Sarandi,Nome,Fechada,US Sarandi
1,09/05/2024,US Sarandi,Situacao,Aberta,Fechada
2,09/05/2024,US Nossa Senhora Aparecida,Cidade,Fechada,Porto Alegre
3,09/05/2024,US Nossa Senhora Aparecida,Situacao,Aberta,Fechada
4,09/05/2024,US Nossa Senhora Aparecida,End_cidade,"Beco José Paris, 235, Sarandi, Fechada","Beco José Paris, 235, Sarandi, Porto Alegre"
5,10/05/2024,US Nossa Senhora Aparecida,Cidade,Fechada,Porto Alegre
6,10/05/2024,US Nossa Senhora Aparecida,Situacao,Aberta,Fechada
7,10/05/2024,US Nossa Senhora Aparecida,End_cidade,"Beco José Paris, 235, Sarandi, Fechada","Beco José Paris, 235, Sarandi, Porto Alegre"


### 5.10 Conferência após os ajustes

Repetimos a correspondência dos nomes nas cópias preparadas, mantendo a análise original acima. A tabela apresenta as duas unidades ajustadas em cada data, para conferir nome, bairro, município, situação adotada e endereço composto. Os valores anteriores e novos estão no registro de alterações.


In [34]:
nomes_preparados_09 = atencao_09_preparada[["Nome"]].rename(columns={"Nome": "nome_09"}).copy()
nomes_preparados_10 = atencao_10_preparada[["name"]].rename(columns={"name": "nome_10"}).copy()
nomes_preparados_09["chave_nome"] = nomes_preparados_09["nome_09"].map(normalizar_nome)
nomes_preparados_10["chave_nome"] = nomes_preparados_10["nome_10"].map(normalizar_nome)

comparacao_atencao_preparada = nomes_preparados_09.merge(
    nomes_preparados_10, on="chave_nome", how="outer",
    indicator=True, validate="one_to_one"
)
display(comparacao_atencao_preparada["_merge"].value_counts().rename_axis("correspondência após ajustes").reset_index(name="unidades"))

,correspondência após ajustes,unidades
0,both,136
1,left_only,0
2,right_only,0


### 5.11 Registros ajustados nas duas versões

Esta saída permite conferir o resultado das decisões. “Fechada” é a situação adotada para US Sarandi e US Nossa Senhora Aparecida, sem confirmação documental da causa ou do funcionamento real.


In [35]:
unidades_ajustadas_atencao = []
for data, camada, campo_nome in [
    ("09/05/2024", atencao_09_preparada, "Nome"),
    ("10/05/2024", atencao_10_preparada, "name"),
]:
    linhas = camada.loc[
        camada[campo_nome].isin(["US Sarandi", "US Nossa Senhora Aparecida"]),
        [campo_nome, "Bairro", "Cidade", "Situacao", "End_cidade"]
    ].rename(columns={campo_nome: "nome"}).copy()
    linhas.insert(0, "data_referencia", data)
    unidades_ajustadas_atencao.append(linhas)
    assert camada["Cidade"].map(normalizar_nome).eq("porto alegre").all()

with pd.option_context("display.max_columns", None, "display.max_colwidth", 60):
    display(pd.concat(unidades_ajustadas_atencao, ignore_index=True))

,data_referencia,nome,Bairro,Cidade,Situacao,End_cidade
0,09/05/2024,US Nossa Senhora Aparecida,Sarandi,Porto Alegre,Fechada,"Beco José Paris, 235, Sarandi, Porto Alegre"
1,09/05/2024,US Sarandi,Sarandi,Porto Alegre,Fechada,"Avenida Francisco Pinto da Fontoura, 341, Sarandi, Porto..."
2,10/05/2024,US Nossa Senhora Aparecida,Sarandi,Porto Alegre,Fechada,"Beco José Paris, 235, Sarandi, Porto Alegre"
3,10/05/2024,US Sarandi,Sarandi,Porto Alegre,Fechada,"Avenida Francisco Pinto da Fontoura, 341, Sarandi, Porto..."


### Estado desta etapa

As versões permanecem separadas, com os ajustes de nome, bairro, município, endereço composto e situação documentados nas cópias de preparação. As situações adotadas por hipótese não comprovam funcionamento real. As camadas originais foram preservadas; a exportação separada das versões preparadas é apresentada a seguir.


### 5.12 Exportação das versões preparadas

As versões são mantidas separadas em `dados/tratados/poa_saude_unidades_atencao_primaria_09052024.parquet` e `dados/tratados/poa_saude_unidades_atencao_primaria_10052024.parquet`, preservando seus atributos e geometrias, com os ajustes documentados anteriormente.

A situação “Fechada” adotada para US Sarandi e US Nossa Senhora Aparecida não constitui confirmação documental do funcionamento ou de sua causa. Os valores anteriores e novos permanecem no registro de alterações do notebook; as camadas do GeoPackage não são modificadas.

A releitura verifica registros, atributos, tipos, valores, CRS e geometrias de cada versão.


In [36]:
DADOS_TRATADOS.mkdir(parents=True, exist_ok=True)

arquivos_atencao_preparada = {
    "09/05/2024": DADOS_TRATADOS / "poa_saude_unidades_atencao_primaria_09052024.parquet",
    "10/05/2024": DADOS_TRATADOS / "poa_saude_unidades_atencao_primaria_10052024.parquet",
}

for data, camada in [
    ("09/05/2024", atencao_09_preparada),
    ("10/05/2024", atencao_10_preparada),
]:
    camada.to_parquet(arquivos_atencao_preparada[data], index=False)
    camada_conferida = gpd.read_parquet(arquivos_atencao_preparada[data])

    pd.testing.assert_frame_equal(camada_conferida, camada.reset_index(drop=True))
    assert camada_conferida.crs == camada.crs, "O CRS não foi preservado."
    assert camada_conferida.geometry.equals(
        camada.geometry.reset_index(drop=True)
    ), "As geometrias não foram preservadas."

    print(f"{data}: {len(camada_conferida)} registros conferidos.")

print("Atributos, tipos, valores, CRS e geometrias preservados nas duas versões.")

09/05/2024: 136 registros conferidos.
10/05/2024: 136 registros conferidos.
Atributos, tipos, valores, CRS e geometrias preservados nas duas versões.


## 6. Inundação em Porto Alegre — manchas históricas estimadas

**Camadas selecionadas:**

- `poa_inundacao_1941_475cm`
- `poa_inundacao_1967_313cm`
- `poa_inundacao_2023_set_318cm`
- `poa_inundacao_2023_nov_346cm`

Foi decidido **não preparar nem selecionar simulações deste GeoPackage**, pois o INCT já dispõe de cenários para Porto Alegre. A exploração é mantida como registro do inventário, sem indicar aproveitamento das simulações nesta etapa.

O glossário descreve as quatro camadas selecionadas como **manchas estimadas de eventos históricos**. A preparação verifica geometrias, repetições, classes, cobertura e medidas de área. A origem das estimativas e a referência das cotas permanecem pendentes; os produtos não são classificados automaticamente como observações diretas.

As camadas de 2015 e 2024 também ficam fora desta etapa: suas descrições no [glossário](../../dados/cheias_rhguaiba_2024_db_v5_glossario.csv) mencionam simulações e cotas divergentes dos nomes. Os produtos HEC-RAS simulados não fazem parte da seleção.

A seção acrescenta `area_ha_calculada`, preserva os atributos originais e exporta as quatro camadas. Uma eventual comparação com os cenários do INCT depende de compatibilidade metodológica e espacial.


### Organização das verificações de inundação

Cada camada é examinada separadamente. As geometrias originais permanecem intactas; somente geometrias não nulas, não vazias, válidas e poligonais participam dos cálculos espaciais. A comparação com os bairros usa sua cobertura disponível, não uma validação definitiva do limite municipal.

As áreas calculadas em metros quadrados são convertidas para hectares dividindo por 10.000. A contagem de geometrias repetidas utiliza WKB exato, incluindo Z quando presente; não detecta toda equivalência topológica nem sobreposições parciais. O diagnóstico não corrige ou remove geometrias. O cálculo da nova coluna de área e a exportação aparecem após as verificações.

**Origem dos polígonos de bairros:** esta seção reutiliza a variável `bairros`, carregada na **seção 2 — Referência territorial**, por `gpd.read_file(ARQUIVO_BANCO, layer="poa_admin_bairros")`. Trata-se dos **83 polígonos da camada original do GeoPackage**, não dos nomes do JSON nem de uma nova base externa.

Os polígonos são unidos em `cobertura_bairros_inundacao`, usada para medir quanto de cada mancha fica dentro ou fora da cobertura disponível. As células de leitura de `bairros` da seção 2 precisam ter sido executadas antes.


**Leitura dos diagnósticos individuais:** a qualidade geométrica separa nulos, vazios e inválidos; duplicatas contam ocorrências excedentes sem removê-las. A frequência de `dn` descreve códigos sem atribuir significado físico. `isna()` não identifica todos os marcadores de informação faltante. A cobertura considera interseção, inclusive bordas, e calcula áreas dentro e fora dos bairros pela união dos polígonos, sem dupla contagem de sobreposições. Nas camadas com `area_ha`, a diferença apresentada é área informada menos área calculada; divergências não determinam sozinhas sua causa.


In [37]:
camadas_inundacao = {
    "1941": "poa_inundacao_1941_475cm",
    "1967": "poa_inundacao_1967_313cm",
    "2023_set": "poa_inundacao_2023_set_318cm",
    "2023_nov": "poa_inundacao_2023_nov_346cm",
}

inundacoes_originais = {}
inundacoes_verificadas = {}

assert bairros.crs is not None
assert bairros.geometry.notna().all() and not bairros.geometry.is_empty.any()
assert bairros.geometry.is_valid.all(), "Investigar os polígonos de bairros antes da comparação."
# Reutiliza os polígonos de poa_admin_bairros carregados na seção 2.
cobertura_bairros_inundacao = bairros.to_crs(31982).geometry.union_all()

### 6.1. 1941 — cota nominal 475 cm: leitura e diagnóstico

Prévia de até dez registros da camada e diagnóstico específico do evento.


In [38]:
chave_inundacao = "1941"
mancha = gpd.read_file(ARQUIVO_BANCO, layer=camadas_inundacao[chave_inundacao])
inundacoes_originais[chave_inundacao] = mancha

print(f"Camada: {camadas_inundacao[chave_inundacao]}")
print(f"Registros: {len(mancha)} | CRS: {mancha.crs}")
with pd.option_context("display.max_columns", None):
    display(mancha.head(10))

Camada: poa_inundacao_1941_475cm
Registros: 248 | CRS: EPSG:31982


,dn,area_ha,geometry
0,1,8,"MULTIPOLYGON (((473660.024 6688037.737, 473696..."
1,1,0,"MULTIPOLYGON (((477739.91 6685465.636, 477743...."
2,1,537,"MULTIPOLYGON (((473988.015 6687869.731, 474000..."
3,1,62,"MULTIPOLYGON (((475875.962 6686145.663, 475895..."
4,1,0,"MULTIPOLYGON (((488627.606 6684837.611, 488631..."
5,1,0,"MULTIPOLYGON (((485127.704 6684749.608, 485179..."
6,1,0,"MULTIPOLYGON (((483575.747 6684601.602, 483587..."
7,1,0,"MULTIPOLYGON (((483603.746 6684605.602, 483643..."
8,1,0,"MULTIPOLYGON (((483483.75 6684585.601, 483511...."
9,1,0,"MULTIPOLYGON (((483475.75 6684581.601, 483483...."


#### Qualidade geométrica e repetições


In [39]:
mancha = inundacoes_originais["1941"]
nulas = mancha.geometry.isna()
vazias = mancha.geometry.is_empty
invalidas = ~nulas & ~vazias & ~mancha.geometry.is_valid
poligonais = mancha.geom_type.isin(["Polygon", "MultiPolygon"])
utilizaveis = ~nulas & ~vazias & ~invalidas & poligonais

repetidas_wkb = mancha.loc[~nulas & ~vazias].geometry.to_wkb().duplicated()

display(pd.Series({
    "registros": len(mancha),
    "geometrias nulas": int(nulas.sum()),
    "geometrias vazias": int(vazias.sum()),
    "geometrias inválidas não vazias": int(invalidas.sum()),
    "geometrias utilizáveis nos cálculos": int(utilizaveis.sum()),
    "linhas inteiramente duplicadas excedentes": int(mancha.duplicated().sum()),
    "geometrias WKB repetidas excedentes": int(repetidas_wkb.sum()),
}, name="quantidade").to_frame())

inundacoes_verificadas["1941"] = {"mascara_utilizavel": utilizaveis}
if (~utilizaveis).any():
    print("Prévia dos registros fora dos cálculos espaciais:")
    display(mancha.loc[~utilizaveis].head(10))

,quantidade
registros,248
geometrias nulas,0
geometrias vazias,0
geometrias inválidas não vazias,0
geometrias utilizáveis nos cálculos,248
linhas inteiramente duplicadas excedentes,0
geometrias WKB repetidas excedentes,0


#### Valores de classe e preenchimento


In [40]:
mancha = inundacoes_originais["1941"]
display(mancha["dn"].value_counts(dropna=False).rename_axis("dn").reset_index(name="registros"))

,dn,registros
0,1,248


#### Ausências por atributo


In [41]:
display(inundacoes_originais["1941"].isna().sum().rename("ausentes").to_frame())

,ausentes
dn,0
area_ha,0
geometry,0


#### Cobertura em relação aos bairros


In [42]:
mancha = inundacoes_originais["1941"]
assert mancha.crs is not None, "A camada precisa ter CRS definido."
utilizaveis = inundacoes_verificadas["1941"]["mascara_utilizavel"]
mancha_calculo = mancha.loc[utilizaveis].to_crs(31982).copy()
inundacoes_verificadas["1941"]["geometrias_calculo"] = mancha_calculo

if not mancha_calculo.empty:
    intersectam = mancha_calculo.geometry.intersects(cobertura_bairros_inundacao)
    uniao_mancha = mancha_calculo.geometry.union_all()
    area_total_ha = uniao_mancha.area / 10000
    area_dentro_ha = uniao_mancha.intersection(cobertura_bairros_inundacao).area / 10000
    area_fora_ha = uniao_mancha.difference(cobertura_bairros_inundacao).area / 10000
    soma_poligonos_ha = mancha_calculo.geometry.area.sum() / 10000
    resumo_cobertura = pd.DataFrame([{
        "polígonos utilizáveis": len(mancha_calculo),
        "intersectam cobertura": int(intersectam.sum()),
        "não intersectam cobertura": int((~intersectam).sum()),
        "área da união (ha)": area_total_ha,
        "área dentro da cobertura (ha)": area_dentro_ha,
        "área fora da cobertura (ha)": area_fora_ha,
        "fora da cobertura (%)": 100 * area_fora_ha / area_total_ha if area_total_ha else None,
        "soma das áreas menos união (ha)": soma_poligonos_ha - area_total_ha,
    }])
    inundacoes_verificadas["1941"]["cobertura"] = resumo_cobertura
    with pd.option_context("display.max_columns", None):
        display(resumo_cobertura)
else:
    print("Nenhuma geometria utilizável para a comparação espacial.")

,polígonos utilizáveis,intersectam cobertura,não intersectam cobertura,área da união (ha),área dentro da cobertura (ha),área fora da cobertura (ha),fora da cobertura (%),soma das áreas menos união (ha)
0,248,187,61,13332.094466,13234.482631,97.611835,0.732157,-1.091394e-11


#### Conferência de `area_ha`

Comparação do atributo original com a área geométrica dos polígonos utilizáveis; zeros e ausências originais são preservados.


In [43]:
mancha_calculo = inundacoes_verificadas["1941"]["geometrias_calculo"]
comparacao_area = pd.DataFrame({
    "area_ha_original": mancha_calculo["area_ha"],
    "area_ha_calculada": mancha_calculo.geometry.area / 10000,
})
comparacao_area["diferenca_ha"] = comparacao_area["area_ha_original"] - comparacao_area["area_ha_calculada"]
comparacao_area["diferenca_absoluta_ha"] = comparacao_area["diferenca_ha"].abs()
inundacoes_verificadas["1941"]["comparacao_area"] = comparacao_area

print(f"Valores originais ausentes na camada: {inundacoes_originais['1941']['area_ha'].isna().sum()}")
print(f"Valores originais iguais a zero na camada: {inundacoes_originais['1941']['area_ha'].eq(0).sum()}")
print(f"Polígonos utilizáveis com área informada: {comparacao_area['area_ha_original'].notna().sum()}")
print("Até dez maiores diferenças absolutas, mantendo o índice original:")
with pd.option_context("display.max_columns", None):
    display(comparacao_area.dropna(subset=["area_ha_original"]).sort_values(
        "diferenca_absoluta_ha", ascending=False
    ).head(10))

Valores originais ausentes na camada: 0
Valores originais iguais a zero na camada: 221
Polígonos utilizáveis com área informada: 248
Até dez maiores diferenças absolutas, mantendo o índice original:


,area_ha_original,area_ha_calculada,diferenca_ha,diferenca_absoluta_ha
247,4773,4768.926873,4.073127,4.073127
246,4074,4070.755720,3.244280,3.244280
57,1613,1611.343372,1.656628,1.656628
59,826,825.457514,0.542486,0.542486
68,22,22.469059,-0.469059,0.469059
99,0,0.462405,-0.462405,0.462405
181,58,57.555957,0.444043,0.444043
20,0,0.428805,-0.428805,0.428805
174,5,4.576053,0.423947,0.423947
10,0,0.417605,-0.417605,0.417605


### 6.2. 1967 — cota nominal 313 cm: leitura e diagnóstico

Prévia de até dez registros da camada e diagnóstico específico do evento.


In [44]:
chave_inundacao = "1967"
mancha = gpd.read_file(ARQUIVO_BANCO, layer=camadas_inundacao[chave_inundacao])
inundacoes_originais[chave_inundacao] = mancha

print(f"Camada: {camadas_inundacao[chave_inundacao]}")
print(f"Registros: {len(mancha)} | CRS: {mancha.crs}")
with pd.option_context("display.max_columns", None):
    display(mancha.head(10))

Camada: poa_inundacao_1967_313cm
Registros: 507 | CRS: EPSG:31982


,dn,geometry
0,1,"MULTIPOLYGON (((473660.024 6688037.737, 473696..."
1,1,"MULTIPOLYGON (((477739.91 6685465.636, 477743...."
2,1,"MULTIPOLYGON (((473988.015 6687869.731, 474000..."
3,1,"MULTIPOLYGON (((475875.962 6686145.663, 475895..."
4,1,"MULTIPOLYGON (((488843.6 6685053.62, 488855.6 ..."
5,1,"MULTIPOLYGON (((488619.606 6684817.611, 488627..."
6,1,"MULTIPOLYGON (((488775.602 6684945.616, 488803..."
7,1,"MULTIPOLYGON (((484147.731 6684709.606, 484155..."
8,1,"MULTIPOLYGON (((484111.732 6684705.606, 484131..."
9,1,"MULTIPOLYGON (((489211.59 6684649.604, 489219...."


#### Qualidade geométrica e repetições


In [45]:
mancha = inundacoes_originais["1967"]
nulas = mancha.geometry.isna()
vazias = mancha.geometry.is_empty
invalidas = ~nulas & ~vazias & ~mancha.geometry.is_valid
poligonais = mancha.geom_type.isin(["Polygon", "MultiPolygon"])
utilizaveis = ~nulas & ~vazias & ~invalidas & poligonais

repetidas_wkb = mancha.loc[~nulas & ~vazias].geometry.to_wkb().duplicated()

display(pd.Series({
    "registros": len(mancha),
    "geometrias nulas": int(nulas.sum()),
    "geometrias vazias": int(vazias.sum()),
    "geometrias inválidas não vazias": int(invalidas.sum()),
    "geometrias utilizáveis nos cálculos": int(utilizaveis.sum()),
    "linhas inteiramente duplicadas excedentes": int(mancha.duplicated().sum()),
    "geometrias WKB repetidas excedentes": int(repetidas_wkb.sum()),
}, name="quantidade").to_frame())

inundacoes_verificadas["1967"] = {"mascara_utilizavel": utilizaveis}
if (~utilizaveis).any():
    print("Prévia dos registros fora dos cálculos espaciais:")
    display(mancha.loc[~utilizaveis].head(10))

,quantidade
registros,507
geometrias nulas,0
geometrias vazias,0
geometrias inválidas não vazias,0
geometrias utilizáveis nos cálculos,507
linhas inteiramente duplicadas excedentes,0
geometrias WKB repetidas excedentes,0


#### Valores de classe e preenchimento


In [46]:
mancha = inundacoes_originais["1967"]
display(mancha["dn"].value_counts(dropna=False).rename_axis("dn").reset_index(name="registros"))

,dn,registros
0,1,507


#### Ausências por atributo


In [47]:
display(inundacoes_originais["1967"].isna().sum().rename("ausentes").to_frame())

,ausentes
dn,0
geometry,0


#### Cobertura em relação aos bairros


In [48]:
mancha = inundacoes_originais["1967"]
assert mancha.crs is not None, "A camada precisa ter CRS definido."
utilizaveis = inundacoes_verificadas["1967"]["mascara_utilizavel"]
mancha_calculo = mancha.loc[utilizaveis].to_crs(31982).copy()
inundacoes_verificadas["1967"]["geometrias_calculo"] = mancha_calculo

if not mancha_calculo.empty:
    intersectam = mancha_calculo.geometry.intersects(cobertura_bairros_inundacao)
    uniao_mancha = mancha_calculo.geometry.union_all()
    area_total_ha = uniao_mancha.area / 10000
    area_dentro_ha = uniao_mancha.intersection(cobertura_bairros_inundacao).area / 10000
    area_fora_ha = uniao_mancha.difference(cobertura_bairros_inundacao).area / 10000
    soma_poligonos_ha = mancha_calculo.geometry.area.sum() / 10000
    resumo_cobertura = pd.DataFrame([{
        "polígonos utilizáveis": len(mancha_calculo),
        "intersectam cobertura": int(intersectam.sum()),
        "não intersectam cobertura": int((~intersectam).sum()),
        "área da união (ha)": area_total_ha,
        "área dentro da cobertura (ha)": area_dentro_ha,
        "área fora da cobertura (ha)": area_fora_ha,
        "fora da cobertura (%)": 100 * area_fora_ha / area_total_ha if area_total_ha else None,
        "soma das áreas menos união (ha)": soma_poligonos_ha - area_total_ha,
    }])
    inundacoes_verificadas["1967"]["cobertura"] = resumo_cobertura
    with pd.option_context("display.max_columns", None):
        display(resumo_cobertura)
else:
    print("Nenhuma geometria utilizável para a comparação espacial.")

,polígonos utilizáveis,intersectam cobertura,não intersectam cobertura,área da união (ha),área dentro da cobertura (ha),área fora da cobertura (ha),fora da cobertura (%),soma das áreas menos união (ha)
0,507,446,61,9688.359668,9591.789647,96.570021,0.996763,1.818989e-12


### 6.3. Setembro de 2023 — cota nominal 318 cm: leitura e diagnóstico

Prévia de até dez registros da camada e diagnóstico específico do evento.


In [49]:
chave_inundacao = "2023_set"
mancha = gpd.read_file(ARQUIVO_BANCO, layer=camadas_inundacao[chave_inundacao])
inundacoes_originais[chave_inundacao] = mancha

print(f"Camada: {camadas_inundacao[chave_inundacao]}")
print(f"Registros: {len(mancha)} | CRS: {mancha.crs}")
with pd.option_context("display.max_columns", None):
    display(mancha.head(10))

Camada: poa_inundacao_2023_set_318cm
Registros: 526 | CRS: EPSG:31982


,dn,area_ha,geometry
0,1,NaN,"MULTIPOLYGON (((473660.024 6688037.737, 473696..."
1,1,NaN,"MULTIPOLYGON (((477739.91 6685465.636, 477743...."
2,1,NaN,"MULTIPOLYGON (((473988.015 6687869.731, 474000..."
3,1,NaN,"MULTIPOLYGON (((475875.962 6686145.663, 475895..."
4,1,NaN,"MULTIPOLYGON (((488843.6 6685053.62, 488855.6 ..."
5,1,NaN,"MULTIPOLYGON (((488619.606 6684817.611, 488627..."
6,1,NaN,"MULTIPOLYGON (((488775.602 6684945.616, 488803..."
7,1,NaN,"MULTIPOLYGON (((484147.731 6684709.606, 484155..."
8,1,NaN,"MULTIPOLYGON (((484111.732 6684705.606, 484131..."
9,1,NaN,"MULTIPOLYGON (((489171.591 6684649.604, 489191..."


#### Qualidade geométrica e repetições


In [50]:
mancha = inundacoes_originais["2023_set"]
nulas = mancha.geometry.isna()
vazias = mancha.geometry.is_empty
invalidas = ~nulas & ~vazias & ~mancha.geometry.is_valid
poligonais = mancha.geom_type.isin(["Polygon", "MultiPolygon"])
utilizaveis = ~nulas & ~vazias & ~invalidas & poligonais

repetidas_wkb = mancha.loc[~nulas & ~vazias].geometry.to_wkb().duplicated()

display(pd.Series({
    "registros": len(mancha),
    "geometrias nulas": int(nulas.sum()),
    "geometrias vazias": int(vazias.sum()),
    "geometrias inválidas não vazias": int(invalidas.sum()),
    "geometrias utilizáveis nos cálculos": int(utilizaveis.sum()),
    "linhas inteiramente duplicadas excedentes": int(mancha.duplicated().sum()),
    "geometrias WKB repetidas excedentes": int(repetidas_wkb.sum()),
}, name="quantidade").to_frame())

inundacoes_verificadas["2023_set"] = {"mascara_utilizavel": utilizaveis}
if (~utilizaveis).any():
    print("Prévia dos registros fora dos cálculos espaciais:")
    display(mancha.loc[~utilizaveis].head(10))

,quantidade
registros,526
geometrias nulas,0
geometrias vazias,0
geometrias inválidas não vazias,0
geometrias utilizáveis nos cálculos,526
linhas inteiramente duplicadas excedentes,0
geometrias WKB repetidas excedentes,0


#### Valores de classe e preenchimento


In [51]:
mancha = inundacoes_originais["2023_set"]
display(mancha["dn"].value_counts(dropna=False).rename_axis("dn").reset_index(name="registros"))

,dn,registros
0,1,526


#### Ausências por atributo


In [52]:
display(inundacoes_originais["2023_set"].isna().sum().rename("ausentes").to_frame())

,ausentes
dn,0
area_ha,316
geometry,0


#### Cobertura em relação aos bairros


In [53]:
mancha = inundacoes_originais["2023_set"]
assert mancha.crs is not None, "A camada precisa ter CRS definido."
utilizaveis = inundacoes_verificadas["2023_set"]["mascara_utilizavel"]
mancha_calculo = mancha.loc[utilizaveis].to_crs(31982).copy()
inundacoes_verificadas["2023_set"]["geometrias_calculo"] = mancha_calculo

if not mancha_calculo.empty:
    intersectam = mancha_calculo.geometry.intersects(cobertura_bairros_inundacao)
    uniao_mancha = mancha_calculo.geometry.union_all()
    area_total_ha = uniao_mancha.area / 10000
    area_dentro_ha = uniao_mancha.intersection(cobertura_bairros_inundacao).area / 10000
    area_fora_ha = uniao_mancha.difference(cobertura_bairros_inundacao).area / 10000
    soma_poligonos_ha = mancha_calculo.geometry.area.sum() / 10000
    resumo_cobertura = pd.DataFrame([{
        "polígonos utilizáveis": len(mancha_calculo),
        "intersectam cobertura": int(intersectam.sum()),
        "não intersectam cobertura": int((~intersectam).sum()),
        "área da união (ha)": area_total_ha,
        "área dentro da cobertura (ha)": area_dentro_ha,
        "área fora da cobertura (ha)": area_fora_ha,
        "fora da cobertura (%)": 100 * area_fora_ha / area_total_ha if area_total_ha else None,
        "soma das áreas menos união (ha)": soma_poligonos_ha - area_total_ha,
    }])
    inundacoes_verificadas["2023_set"]["cobertura"] = resumo_cobertura
    with pd.option_context("display.max_columns", None):
        display(resumo_cobertura)
else:
    print("Nenhuma geometria utilizável para a comparação espacial.")

,polígonos utilizáveis,intersectam cobertura,não intersectam cobertura,área da união (ha),área dentro da cobertura (ha),área fora da cobertura (ha),fora da cobertura (%),soma das áreas menos união (ha)
0,526,460,66,9813.321108,9716.687586,96.633522,0.984718,1.455192e-11


#### Conferência de `area_ha`

Comparação do atributo original com a área geométrica dos polígonos utilizáveis; zeros e ausências originais são preservados.


In [54]:
mancha_calculo = inundacoes_verificadas["2023_set"]["geometrias_calculo"]
comparacao_area = pd.DataFrame({
    "area_ha_original": mancha_calculo["area_ha"],
    "area_ha_calculada": mancha_calculo.geometry.area / 10000,
})
comparacao_area["diferenca_ha"] = comparacao_area["area_ha_original"] - comparacao_area["area_ha_calculada"]
comparacao_area["diferenca_absoluta_ha"] = comparacao_area["diferenca_ha"].abs()
inundacoes_verificadas["2023_set"]["comparacao_area"] = comparacao_area

print(f"Valores originais ausentes na camada: {inundacoes_originais['2023_set']['area_ha'].isna().sum()}")
print(f"Valores originais iguais a zero na camada: {inundacoes_originais['2023_set']['area_ha'].eq(0).sum()}")
print(f"Polígonos utilizáveis com área informada: {comparacao_area['area_ha_original'].notna().sum()}")
print("Até dez maiores diferenças absolutas, mantendo o índice original:")
with pd.option_context("display.max_columns", None):
    display(comparacao_area.dropna(subset=["area_ha_original"]).sort_values(
        "diferenca_absoluta_ha", ascending=False
    ).head(10))

Valores originais ausentes na camada: 316
Valores originais iguais a zero na camada: 0
Polígonos utilizáveis com área informada: 210
Até dez maiores diferenças absolutas, mantendo o índice original:


,area_ha_original,area_ha_calculada,diferenca_ha,diferenca_absoluta_ha
525,753.486,752.885478,0.600522,0.600522
411,514.582,514.175526,0.406474,0.406474
522,474.182,473.805461,0.376539,0.376539
491,360.659,360.373754,0.285246,0.285246
369,67.848,67.794381,0.053619,0.053619
326,36.035,36.006815,0.028185,0.028185
499,12.301,12.291342,0.009658,0.009658
438,9.122,9.115305,0.006695,0.006695
507,5.814,5.809667,0.004333,0.004333
404,3.209,3.206437,0.002563,0.002563


### 6.4. Novembro de 2023 — cota nominal 346 cm: leitura e diagnóstico

Prévia de até dez registros da camada e diagnóstico específico do evento.


In [55]:
chave_inundacao = "2023_nov"
mancha = gpd.read_file(ARQUIVO_BANCO, layer=camadas_inundacao[chave_inundacao])
inundacoes_originais[chave_inundacao] = mancha

print(f"Camada: {camadas_inundacao[chave_inundacao]}")
print(f"Registros: {len(mancha)} | CRS: {mancha.crs}")
with pd.option_context("display.max_columns", None):
    display(mancha.head(10))

Camada: poa_inundacao_2023_nov_346cm
Registros: 424 | CRS: EPSG:31982


,dn,geometry
0,1,"POLYGON ((473660.024 6688037.737, 473660.024 6..."
1,1,"POLYGON ((477739.91 6685465.636, 477739.91 668..."
2,1,"POLYGON ((473988.015 6687869.731, 473988.015 6..."
3,1,"POLYGON ((475875.962 6686145.663, 475875.962 6..."
4,1,"POLYGON ((488847.6 6685061.62, 488847.6 668505..."
5,1,"POLYGON ((488623.606 6684825.611, 488623.606 6..."
6,1,"POLYGON ((484143.731 6684709.606, 484143.731 6..."
7,1,"POLYGON ((484107.732 6684705.606, 484107.732 6..."
8,1,"POLYGON ((483615.746 6684605.602, 483615.746 6..."
9,1,"POLYGON ((489175.591 6684653.604, 489175.591 6..."


#### Qualidade geométrica e repetições


In [56]:
mancha = inundacoes_originais["2023_nov"]
nulas = mancha.geometry.isna()
vazias = mancha.geometry.is_empty
invalidas = ~nulas & ~vazias & ~mancha.geometry.is_valid
poligonais = mancha.geom_type.isin(["Polygon", "MultiPolygon"])
utilizaveis = ~nulas & ~vazias & ~invalidas & poligonais

repetidas_wkb = mancha.loc[~nulas & ~vazias].geometry.to_wkb().duplicated()

display(pd.Series({
    "registros": len(mancha),
    "geometrias nulas": int(nulas.sum()),
    "geometrias vazias": int(vazias.sum()),
    "geometrias inválidas não vazias": int(invalidas.sum()),
    "geometrias utilizáveis nos cálculos": int(utilizaveis.sum()),
    "linhas inteiramente duplicadas excedentes": int(mancha.duplicated().sum()),
    "geometrias WKB repetidas excedentes": int(repetidas_wkb.sum()),
}, name="quantidade").to_frame())

inundacoes_verificadas["2023_nov"] = {"mascara_utilizavel": utilizaveis}
if (~utilizaveis).any():
    print("Prévia dos registros fora dos cálculos espaciais:")
    display(mancha.loc[~utilizaveis].head(10))

,quantidade
registros,424
geometrias nulas,0
geometrias vazias,0
geometrias inválidas não vazias,0
geometrias utilizáveis nos cálculos,424
linhas inteiramente duplicadas excedentes,0
geometrias WKB repetidas excedentes,0


#### Valores de classe e preenchimento


In [57]:
mancha = inundacoes_originais["2023_nov"]
display(mancha["dn"].value_counts(dropna=False).rename_axis("dn").reset_index(name="registros"))

,dn,registros
0,1,424


#### Ausências por atributo


In [58]:
display(inundacoes_originais["2023_nov"].isna().sum().rename("ausentes").to_frame())

,ausentes
dn,0
geometry,0


#### Cobertura em relação aos bairros


In [59]:
mancha = inundacoes_originais["2023_nov"]
assert mancha.crs is not None, "A camada precisa ter CRS definido."
utilizaveis = inundacoes_verificadas["2023_nov"]["mascara_utilizavel"]
mancha_calculo = mancha.loc[utilizaveis].to_crs(31982).copy()
inundacoes_verificadas["2023_nov"]["geometrias_calculo"] = mancha_calculo

if not mancha_calculo.empty:
    intersectam = mancha_calculo.geometry.intersects(cobertura_bairros_inundacao)
    uniao_mancha = mancha_calculo.geometry.union_all()
    area_total_ha = uniao_mancha.area / 10000
    area_dentro_ha = uniao_mancha.intersection(cobertura_bairros_inundacao).area / 10000
    area_fora_ha = uniao_mancha.difference(cobertura_bairros_inundacao).area / 10000
    soma_poligonos_ha = mancha_calculo.geometry.area.sum() / 10000
    resumo_cobertura = pd.DataFrame([{
        "polígonos utilizáveis": len(mancha_calculo),
        "intersectam cobertura": int(intersectam.sum()),
        "não intersectam cobertura": int((~intersectam).sum()),
        "área da união (ha)": area_total_ha,
        "área dentro da cobertura (ha)": area_dentro_ha,
        "área fora da cobertura (ha)": area_fora_ha,
        "fora da cobertura (%)": 100 * area_fora_ha / area_total_ha if area_total_ha else None,
        "soma das áreas menos união (ha)": soma_poligonos_ha - area_total_ha,
    }])
    inundacoes_verificadas["2023_nov"]["cobertura"] = resumo_cobertura
    with pd.option_context("display.max_columns", None):
        display(resumo_cobertura)
else:
    print("Nenhuma geometria utilizável para a comparação espacial.")

,polígonos utilizáveis,intersectam cobertura,não intersectam cobertura,área da união (ha),área dentro da cobertura (ha),área fora da cobertura (ha),fora da cobertura (%),soma das áreas menos união (ha)
0,424,363,61,11305.587908,11208.505625,97.082283,0.858711,-7.457857e-11


### Encerramento das verificações de inundação

As camadas completas permanecem em `inundacoes_originais`; máscaras e resultados de conferência estão em `inundacoes_verificadas`. Nenhuma geometria foi corrigida ou removida. A etapa seguinte acrescenta a área calculada em cópias de preparação, sem substituir os atributos originais.

### Área calculada para as quatro manchas

Foi decidido criar **`area_ha_calculada`** nas quatro camadas para disponibilizar uma medida obtida pelo mesmo procedimento. O cálculo usa as geometrias em EPSG:31982 e converte metros quadrados para hectares, dividindo por 10.000.

As cópias em `inundacoes_preparadas` mantêm todos os registros, atributos, CRS e geometrias originais. `area_ha` permanece intacto onde existe; não é preenchido nem substituído. Registros sem geometria utilizável permaneceriam sem área calculada, sem exclusão.

A medida descreve cada polígono disponível: não confirma a precisão da mancha ou a extensão efetivamente inundada no evento. A soma por registro não elimina eventuais sobreposições; para área total sem dupla contagem, a comparação anterior utiliza a união das geometrias. A exportação e a conferência dos arquivos aparecem na etapa seguinte.

In [60]:
inundacoes_preparadas = {}
resumo_areas_calculadas = []

for referencia, original in inundacoes_originais.items():
    preparada = original.copy()
    preparada["area_ha_calculada"] = float("nan")
    utilizaveis = inundacoes_verificadas[referencia]["mascara_utilizavel"]

    preparada.loc[utilizaveis, "area_ha_calculada"] = (
        original.loc[utilizaveis].to_crs(31982).geometry.area / 10000
    )

    pd.testing.assert_frame_equal(preparada[original.columns], original)
    assert preparada.crs == original.crs, "O CRS original foi alterado."
    assert preparada.geometry.equals(original.geometry), "As geometrias foram alteradas."
    assert preparada.loc[utilizaveis, "area_ha_calculada"].gt(0).all(), "Investigar áreas não positivas."
    assert preparada.loc[~utilizaveis, "area_ha_calculada"].isna().all()

    inundacoes_preparadas[referencia] = preparada
    resumo_areas_calculadas.append({
        "referência": referencia,
        "registros preservados": len(preparada),
        "áreas calculadas": int(preparada["area_ha_calculada"].notna().sum()),
        "áreas não calculadas": int(preparada["area_ha_calculada"].isna().sum()),
        "possui area_ha original": "area_ha" in original.columns,
        "soma por polígono (ha)": preparada["area_ha_calculada"].sum(min_count=1),
    })

with pd.option_context("display.max_columns", None):
    display(pd.DataFrame(resumo_areas_calculadas))

,referência,registros preservados,áreas calculadas,áreas não calculadas,possui area_ha original,soma por polígono (ha)
0,1941,248,248,0,True,13332.094466
1,1967,507,507,0,False,9688.359668
2,2023_set,526,526,0,True,9813.321108
3,2023_nov,424,424,0,False,11305.587908


### Prévia das áreas preparadas

Cinco registros de cada camada permitem conferir a nova coluna e compará-la ao atributo original, quando presente. As geometrias continuam disponíveis nas cópias preparadas, mas são omitidas desta prévia para facilitar a leitura.

In [61]:
previas_areas_inundacao = []
for referencia, preparada in inundacoes_preparadas.items():
    previa = preparada.head(5).drop(columns="geometry").copy()
    previa.insert(0, "referencia", referencia)
    previas_areas_inundacao.append(previa)

with pd.option_context("display.max_rows", 20, "display.max_columns", None):
    display(pd.concat(previas_areas_inundacao, ignore_index=True))

,referencia,dn,area_ha,area_ha_calculada
0,1941,1,8.0,8.284895
1,1941,1,0.0,0.001600
2,1941,1,537.0,536.801387
3,1941,1,62.0,61.626310
4,1941,1,0.0,0.324804
5,1967,1,NaN,8.284895
6,1967,1,NaN,0.001600
7,1967,1,NaN,536.793387
8,1967,1,NaN,61.626310
9,1967,1,NaN,0.512006


### Exportação e conferência das quatro manchas

Foi decidido salvar um GeoParquet por camada em `dados/tratados/`, utilizando o nome original da camada. Os arquivos preservam `dn`, `geometry`, `area_ha_calculada` e `area_ha` onde já existia. Os polígonos não são recortados pela cobertura dos bairros.

A releitura confere quantidade de registros, atributos, tipos, valores, CRS e geometrias. A interpretação de `dn = 1` e as limitações metodológicas das manchas históricas permanecem pendentes; a exportação não resolve essas questões. O GeoPackage original permanece intacto.

In [62]:
DADOS_TRATADOS.mkdir(parents=True, exist_ok=True)
arquivos_inundacoes_preparadas = {}
conferencias_exportacao_inundacoes = []

for referencia, preparada in inundacoes_preparadas.items():
    arquivo_saida = DADOS_TRATADOS / f"{camadas_inundacao[referencia]}.parquet"
    preparada.to_parquet(arquivo_saida, index=False)
    conferida = gpd.read_parquet(arquivo_saida)

    pd.testing.assert_frame_equal(conferida, preparada.reset_index(drop=True))
    assert conferida.crs == preparada.crs, "O CRS não foi preservado."
    assert conferida.geometry.equals(
        preparada.geometry.reset_index(drop=True)
    ), "As geometrias não foram preservadas."

    arquivos_inundacoes_preparadas[referencia] = arquivo_saida
    conferencias_exportacao_inundacoes.append({
        "referência": referencia,
        "registros conferidos": len(conferida),
        "áreas calculadas": int(conferida["area_ha_calculada"].notna().sum()),
        "conferência": "OK",
    })

display(pd.DataFrame(conferencias_exportacao_inundacoes))
print("Atributos, tipos, valores, CRS e geometrias preservados nas quatro versões.")

,referência,registros conferidos,áreas calculadas,conferência
0,1941,248,248,OK
1,1967,507,507,OK
2,2023_set,526,526,OK
3,2023_nov,424,424,OK


Atributos, tipos, valores, CRS e geometrias preservados nas quatro versões.


## 7. Síntese da preparação

As decisões, justificativas e verificações estão documentadas nas respectivas etapas. Foram gerados **nove arquivos GeoParquet** em `dados/tratados/`: bairros, escolas, cadastro SUS, duas versões de atenção primária e quatro manchas históricas estimadas.

Os arquivos foram relidos para conferir registros, atributos, tipos, valores, CRS e geometrias em relação às versões preparadas. As fontes originais foram preservadas. As limitações e as situações adotadas por hipótese permanecem registradas ao longo do notebook e devem ser consideradas em eventuais análises futuras.
